# 🎬 MiniMax H3｜繁體中文・表單＋Gradio 面板版

在 Colab 上用 ComfyUI 跑 MiniMax H3 影片生成。可以用 **Colab 表單**（STEP 6）、**Gradio 圖形介面**（STEP 6G）或 **JSON 批次**（STEP 7）三種方式操作。

## 使用前請先知道

- **GPU**：請選 L4 或 A100。T4 幾乎跑不動。
- **雲端硬碟空間**：所有檔案都放在 `我的雲端硬碟/AI_Video/`。模型會存兩份（下載快取一份、ComfyUI 用一份），不含參考素材模型約需 80 GB，勾選 `ENABLE_REF2VID` 約需 120 GB。第一次執行 STEP 2 會下載很久。
- **免費版 Colab**：Colab 規定免費執行階段不能主要透過網頁介面操作，使用 STEP 6G 圖形介面可能被中斷。付費方案且運算單元有餘額時不受此限。STEP 6 表單不受影響。
- **運算單元**：執行階段開著就會持續扣。用完請到「執行階段 → 中斷連線並刪除執行階段」。

## 每次重開 Colab 請按順序

1. **STEP 1**：掛載 Google Drive、安裝環境、選模式。
2. **STEP 2**：下載／檢查模型。要用參考素材生成（r2v）請勾選 `ENABLE_REF2VID`。
3. **STEP 3**：啟動 ComfyUI。如果顯示「10 分鐘內沒有回應」，直接再執行一次這格。
4. **STEP 5**：**必要！** 載入工作流程與共用工具（不會生成影片）。
5. 接著三選一：
   - **STEP 6**：用 Colab 表單生成單支影片。
   - **STEP 6G**：開啟 Gradio 圖形介面。
   - **STEP 7**：用 JSON 批次生成多支影片。

小技巧：點一下 STEP 5.5 那格，按 `Ctrl+F8`（執行上方所有儲存格）就會依序跑完 STEP 1 到 5。**不要按「全部執行」**，它會連 STEP 7 和會真的生成影片的 STEP 8.5 一起跑。

## 三種生成方式

- `fl2va`（首尾幀生成）：給一張首幀，讓畫面直接動起來；尾幀可選填。
- `r2v`（參考素材生成）：給角色或風格的參考圖，讓模型重新生成場景。提示詞用 `<Picture 1>`、`<Picture 2>` 對應參考圖的順序。
- `t2va`（純文字生成）：不用任何圖片。

## 圖片與成品放在哪裡

- 參考圖放在 `AI_Video/H3_Reference_Images/`。可以用 STEP 5.5 從電腦上傳，或直接把圖放進雲端硬碟的這個資料夾。
- STEP 6 的圖片欄位可以**只填檔名**，會自動到上面的資料夾找；貼完整路徑也可以。
- 成品會複製到 `AI_Video/Outputs/`，旁邊有同名 `.txt` 生成紀錄（完整提示詞、seed、參數）。

## 注意事項

- 提示詞請用英文。STEP 6 的 `PROMPT` 是單行欄位，直接貼上即可，不需要換行。
- 秒數是整數；想要 9.6 秒可以先生成 10 秒再後製裁剪。
- r2v 固定使用基礎模型，不套用 STEP 1 選的加速模式，會比較慢。
- STEP 7 的 r2v 批次尚未實際用 GPU 測過，第一次請先少量測試。
- STEP 8.5 會真的執行推論並消耗 GPU 額度，只在除錯時使用。
- 無法保證在所有環境或模型節點下都沒有問題；上游更新時，套件與節點的相容性需要重新比對。

## 來源與授權

本 notebook 修改自 Skquark 的 AEI-Colab-Notebooks：
<https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-H3_ComfyUI_Colab.ipynb>

Notebook 程式碼沿用原作的 MIT License（條文如下）。MIT 只涵蓋 notebook 程式碼；模型權重另有各自的授權，使用前請自行確認。

```
MIT License

Copyright (c) 2026 Skquark

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```

### STEP 1｜安裝與掛載 Google 雲端硬碟
執行這格。首次安裝 ComfyUI 及相依套件，會顯示許多英文日誌；若沒有錯誤，繼續下一格。請保留原有 GPU 加速參數。

In [ ]:
#@title STEP 1｜安裝 ComfyUI 與套件

"""
• Mounts Google Drive for the weights cache + ComfyUI installation
• Clones ComfyUI v0.30.1+ to /content/drive/MyDrive/AI_Video/AEI_ComfyUI/ on first run
• Installs torch 2.11.0+cu130 (L4 / T4 / A100 compatible)
• Installs ComfyUI's requirements.txt (transformers, tokenizers, safetensors, av, ...)
• Sets PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to reduce fragmentation
"""

import os, sys, subprocess, time, pathlib
from pathlib import Path

print('='*72)
print('MiniMax-H3 / ComfyUI — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/AI_Video')
else:
    DRIVE_ROOT = Path('/content/_h3_cache')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

COMFY_DIR = DRIVE_ROOT / 'AEI_ComfyUI'
HF_CACHE  = DRIVE_ROOT / 'AEI_3D_Cache' / 'H3_ComfyUI'
OUT_DIR   = DRIVE_ROOT / 'AEI_3D_Out' / 'MiniMax-H3'
COMFY_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME']               = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE']  = str(HF_CACHE)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print(f'  Drive cache  : {HF_CACHE}')
print(f'  ComfyUI dir  : {COMFY_DIR}')
print(f'  Output dir   : {OUT_DIR}')

if not COMFY_DIR.joinpath('main.py').exists():
    print(f'  Cloning ComfyUI to {COMFY_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
else:
    print(f'  Reusing existing {COMFY_DIR}')

# === Mode picker (printed below) ===
# Each mode is one of the 29 dropdown options. Composition rules:
#   - First prefix (optional): diffusion precision — `fp8_dense+`
#   - Second prefix (optional): optimizer  — `spectrum+`
#   - Suffix (required): model flavor — `standard`, `turbo_lora`,
#     `turbo_lora_sage`, `lightx2v_4step`, `lightx2v_8step`,
#     `lightx2v_4step_ersde`, `lightx2v_8step_ersde`,
#     `lightx2v_4step_768p`
# The `_ersde` suffix swaps sampler to er_sde and LoRA strength to 0.75
# (upstream's recipe for lightx2v). The `lightx2v_4step_768p` suffix
# swaps to the 768p-trained v1.0 LoRA with σ shift 6/3 (different from
# 544p variants) — no `_ersde` or composed variants for 768p since
# upstream only recommends the standalone recipe. All other combos are
# legal with the exclusion of `spectrum+standard` (Spectrum without
# SigmaShift is not in the eval).
#
# Recommended picks:
#   L4+Pro+ (24 GB VRAM, 50+ GB host RAM):
#     - lightx2v_8step               — eval's "lx8-house"; 8 steps, ~2-3 min/5s clip
#     - spectrum+turbo_lora_sage     — eval's "emck8-sage-spec" best config; 10 steps, ~3-4 min
#     - fp8_dense+lightx2v_8step     — same speed as lightx2v_8step, cleaner fp8 numerics
#   T4 (16 GB VRAM, 12-25 GB host RAM):
#     - turbo_lora_sage              — Sage saves attention memory; 8 steps; CPU offload OK
#     - standard                     — 20 steps, no LoRA, slowest but safest
#   Drive cost (incremental, on top of base 40.8 GB):
#     - turbo_lora:    +0.7 GB
#     - lightx2v_*:    +1.96 GB (v1.0 8-step 544p OR v1.0 4-step 768p)
#     - both:          +2.66 GB
print('='*72)
print('  Mode picker (T4 is CPU-offload bound; switch to L4 for serious work)')
print('='*72)
print('  | mode                          | steps | sampler     | strength | L4    | T4    |')
print('  |-------------------------------|-------|-------------|----------|-------|-------|')
print('  | standard                       |   20  | res_multistep|    -     | ★★★★  | ✗     |')
print('  | turbo_lora                     |   10  | res_multistep|   1.00   | ★★★★  | ✗     |')
print('  | turbo_lora_sage                |    8  | res_multistep|   1.00   | ★★★★  | ✗     |')
print('  | spectrum                       |   16  | res_multistep|    -     | ★★★★  | ✗     |')
print('  | spectrum+turbo_lora            |   12  | res_multistep|   1.00   | ★★★★  | ✗     |')
print('  | spectrum+turbo_lora_sage       |   10  | res_multistep|   1.00   | ★★★★★| ✗     |  ← best quality')
print('  | fp8_dense                      |   16  | res_multistep|    -     | ★★★★  | ✗     |')
print('  | fp8_dense+lightx2v_8step       |    8  | res_multistep|   1.00   | ★★★★★| ✗     |  ← best speed')
print('  | lightx2v_4step                 |    4  | res_multistep|   1.00   | ★★★★  | ★     |')
print('  | lightx2v_8step                 |    8  | res_multistep|   1.00   | ★★★★  | ✗     |')
print('  | lightx2v_4step_768p            |    4  | res_multistep|   1.00   | ★★★★  | ✗ (768p)|')
print('  | lightx2v_4step_ersde           |    5  | er_sde      |   0.75   | ★★★★  | -     |')
print('  | lightx2v_8step_ersde           |    9  | er_sde      |   0.75   | ★★★★  | -     |')
print('  | ... 16 more combos via fp8_dense+ and spectrum+ prefixes')
print('='*72)
print('  Measured on T4: 2h 15m per image with turbo_lora_sage (8 steps).')
print('  Bottleneck is CPU↔GPU weight streaming; the 19.5 GB int8 diffusion')
print('  exceeds the T4 16 GB VRAM. The only mode that helps on T4 is lightx2v_4step')
print('  (fewer sampling steps; ~1h per image estimated). For serious work, switch')
print('  to L4 — the 121 vs 65 TFLOPS FP16 gap matters more than any mode picker.')

# Optimization mode. Each mode = a model-precision prefix (optional)
# + an optional `spectrum+` token + a turbo/lightx2v suffix. Standard
# is the Comfy-Org/MiniMax-H3 int8_convrot base at 20 steps. The turbo
# paths (turbo_lora, turbo_lora_sage) add the drbaph EMA-ckpt500 LoRA
# + SigmaShift(12, 6) and drop steps. Sage adds the kjnodes
# MemoryEfficientAttentionPatch. fp8_dense swaps the int8_convrot
# diffusion for the fp8_scaled checkpoint (cleaner numerics). spectrum
# inserts xmarre/ComfyUI-Spectrum-MiniMax-H3's Spectral Apply node
# between SigmaShift and the guider (Chebyshev-ridge forecasting of
# post-transformer features, ~25-30% wallclock reduction on H3).
# The lightx2v paths load lightx2v's v1.0 distillation LoRAs from
# lightx2v/Minimax-h3-Turbo. Two ComfyUI-target files: the 8-step 544p
# (used for lightx2v_4step / lightx2v_8step / _ersde variants — upstream
# confirms NFE=4 and NFE=8 both work) and the 4-step 768p (used only
# for lightx2v_4step_768p, with a different σ shift trained on a
# 1344x768 base). Both load via plain LoraLoaderModelOnly. The
# `_ersde` suffix swaps the sampler to er_sde and drops the LoRA strength
# to 0.75 (upstream's documented recipe; matches the eval's lx4-kijai
# column).
# Mode is consumed by STEPs 2/3/4/6/7.
#
# Composition rules.
#   - First prefix is always a diffusion-precision switch: `fp8_dense+`
#   - Second prefix is the optimization: `spectrum+` (spectrum needs
#     SigmaShift downstream of LoRA, so spectrum+turbo is the only
#     spectrum compose we expose)
#   - Suffix is the turbo/lightx2v path. Rightmost `+` segment wins.
#   - All combinations are legal with the exclusion of `spectrum+standard` (Spectrum
#     without a SigmaShift can still forecast, but the eval doesn't
#     cover that case and our schedule assumes SigmaShift is upstream)
MODE = 'standard'  #@param ['standard', 'turbo_lora', 'turbo_lora_sage', 'spectrum', 'spectrum+turbo_lora', 'spectrum+turbo_lora_sage', 'fp8_dense', 'fp8_dense+turbo_lora', 'fp8_dense+turbo_lora_sage', 'fp8_dense+spectrum', 'fp8_dense+spectrum+turbo_lora', 'fp8_dense+spectrum+turbo_lora_sage', 'lightx2v_4step', 'lightx2v_8step', 'lightx2v_4step_ersde', 'lightx2v_8step_ersde', 'lightx2v_4step_768p', 'spectrum+lightx2v_4step', 'spectrum+lightx2v_8step', 'spectrum+lightx2v_4step_ersde', 'spectrum+lightx2v_8step_ersde', 'fp8_dense+lightx2v_4step', 'fp8_dense+lightx2v_8step', 'fp8_dense+lightx2v_4step_ersde', 'fp8_dense+lightx2v_8step_ersde', 'fp8_dense+spectrum+lightx2v_4step', 'fp8_dense+spectrum+lightx2v_8step', 'fp8_dense+spectrum+lightx2v_4step_ersde', 'fp8_dense+spectrum+lightx2v_8step_ersde'] {"allow-input": true}

# Turbo backend. The 'turbo_lora' / 'turbo_lora_sage' modes support two
# different LoRA + node-graph implementations.
#   - drbaph:    drbaph/MiniMax-H3-Turbo-Lora-ComfyUI's
#                minimax_h3_turbo_4step_ema_ckpt500_pruned_comfyui.safetensors,
#                applied via LoraLoaderModelOnly + MiniMaxH3SigmaShift +
#                KSamplerSelect (res_multistep). Sharpest, fastest on
#                roomy GPUs (L4+, A100). The path we've shipped since v2.
#   - larryvrh:  larryvrh/MiniMax-H3-Turbo-LoRA's
#                minimax_h3_turbo_v4_step600_ema.safetensors, applied via
#                the Larryvrh custom node's MiniMaxH3TurboLoRA +
#                MiniMaxH3TurboSampler. Auto-detects our pruned/int8 base
#                and re-injects the LoRA's adaln update at run time. The
#                Turbo Sampler handles the audio/video dual-schedule
#                internally, so we don't need our SigmaShift node in
#                this path. Recommended for sub-16 GB VRAM hosts (T4).
#                The upstream reference workflow uses 6 steps + this
#                LoRA at strength 1.0.
# Either backend uses the same mode name (e.g. turbo_lora_sage); the
# backend only changes the workflow graph + the LoRA file + the
# required custom nodes.
TURBO_BACKEND = 'drbaph'  #@param ['drbaph', 'larryvrh'] {"allow-input": true}

# Low-VRAM mode for larryvrh backend. When True, the Larryvrh Turbo LoRA
# node folds the LoRA into the base weights (cheaper on VRAM, slightly
# softer on quantized bases — the delta is partly rounded away). Default
# False (bypass mode = sharpest output, costs more peak VRAM). On a T4
# with 16 GB, set True to free ~1-2 GB of peak; on an L4+ leave False.
LOW_VRAM = False  #@param {type:"boolean"}

# ComfyUI launch flags (`--reserve-vram`, `--fast-disk`,
# `--use-sage-attention`) are auto-detected in STEP 3 from the GPU's
# compute capability. Turing (sm_75) gets `--reserve-vram 0.9
# --fast-disk`; Ampere/Ada/Hopper get no extra flags. The
# `--use-sage-attention` flag path is dead (it required a form param
# that was removed in commit 82119de and the references to it crashed
# STEP 1 with NameError; the kjnodes SageAttention node covers the
# same kernel with fewer install issues, gated on `ATTENTION=sage`).
# Attention backend. `sage` (default) is the kjnodes MiniMax H3 Sage
# Attention Patch — ~30% speedup, validated for sm_80+. `sol` uses
# Saganaki22/ComfyUI-sol-attn's H3 Memory-Efficient Sol Attention
# Patch — measured 1.14-1.44x over Sage on H3, with the same quality
# (Sol is a content-dependent routing algorithm that skips KV blocks
# that don't contribute; the model still sees the un-skipped blocks
# so the output is within noise of the dense path). `sage+sol`
# chains both: Sage is the global fallback, Sol handles eligible
# steps. `none` skips the attention patch entirely (PyTorch SDPA).
# `t4_lowmem` is the kjnodes `MiniMaxLowVRAMAttention` node — splits
# the attention call into head groups so kernel-internal transients
# (int8 q/k copies, fp32 accumulator) shrink by the chunk count while
# the fused qkv buffer is alive. Output is bit-exact with the unpatched
# attention. Composes with ComfyUI's built-in `optimized_attention`
# (cuDNN flash-attention 4 on sm_80+, cuDNN regular on sm_75), so it
# works on every GPU — including Turing T4 where Sage/Sol refuse to
# compile. Recommended for T4 (16 GB VRAM); pair with `spectrum+` to
# cut both attention AND feedforward peak memory. On L4/A100 the node
# is unnecessary (the model fits in VRAM at native attention) but
# harmless; on T4 it makes the difference between OOM and a usable
# pipeline at 30-50% of L4 wallclock.
ATTENTION = 'sage'  #@param ['none', 'sage', 'sol', 'sage+sol', 't4_lowmem'] {"allow-input": true}

# Sampler dropdown. res_multistep is the default for drbaph LoRA and
# native modes; er_sde is the sampler Kijai documents for lightx2v at
# 0.75 strength. The user can override; the override is stored in
# H3_SAMPLER and consumed by STEP 5 / 6 / 7 regardless of MODE.
# NOTE: when TURBO_BACKEND = 'larryvrh' AND MODE is a turbo path,
# this dropdown is IGNORED — the Larryvrh Turbo Sampler hardcodes its
# own 4-step Euler-style sampler and emits it as the SAMPLER output.
# The Turbo Sampler handles audio/video dual-schedule internally, so
# SigmaShift is also skipped in the larryvrh path.
#
# NOTE: Spectrum's allowlist is `euler`, `res_multistep`, `res_multistep_cfg_pp`.
# er_sde is NOT in that allowlist. If SAMPLER = er_sde and the spectrum
# segment is set, the workflow sets Spectrum's `enabled=False` (Spectrum
# auto-disables for non-allowlist samplers). Users running
# spectrum+lightx2v_*_ersde still get the sampler-floor of actual native
# H3 evaluations plus Spectrum's adaptive schedule, which is faster than
# vanilla 20-step native.
#
# The default is mode-driven: anything ending in `_ersde` uses er_sde,
# everything else uses res_multistep. We resolve the default to a literal
# string above the form param line so Colab's form parser doesn't see the
# conditional expression as the dropdown's default value.
SAMPLER = 'res_multistep'  #@param ['res_multistep', 'er_sde', 'euler', 'res_multistep_cfg_pp'] {"allow-input": true}
if SAMPLER == 'res_multistep' and MODE.rsplit('+', 1)[-1].endswith('_ersde'):
    SAMPLER = 'er_sde'

print('  Installing pytorch cu130 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'torch', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu130'], check=False)

print('  Installing ComfyUI requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-r', str(COMFY_DIR / 'requirements.txt')], check=False)

# The requirements pull in comfy-kitchen + comfy-aimdo which are no-op stubs for the
# MiniMax-H3 path. We don't need torchao for INT8 (ComfyUI handles it natively).

import torch as _torch_check
print(f'  torch        : {_torch_check.__version__}  (CUDA {_torch_check.version.cuda})')
if _torch_check.cuda.is_available():
    p = _torch_check.cuda.get_device_properties(0)
    print(f'  GPU          : {p.name}  ({p.total_memory / 1024**3:.1f} GB)')
    print(f'  Compute      : {p.major}.{p.minor}')
else:
    raise SystemExit('No GPU detected - ComfyUI needs CUDA.')

# Persist MODE for the rest of the cells (STEPs 2/3/4/6/7 all consume it).
# We use a global dict so that downstream cells can read MODE without
# having to import it (the previous H3_* globals pattern).
_MODE_GLOBAL = globals().setdefault('_H3_MODE', MODE)
import builtins as _builtins
_builtins.H3_MODE = MODE
_builtins.H3_SAMPLER = SAMPLER
_builtins.H3_TURBO_BACKEND = TURBO_BACKEND
_builtins.H3_LOW_VRAM = LOW_VRAM
_builtins.H3_ATTENTION = ATTENTION
# ComfyUI launch flags (--reserve-vram / --fast-disk) are now auto-detected
# in STEP 3 from the GPU's compute capability (Turing sm_75 → --reserve-vram
# 0.9 --fast-disk; Ampere/Ada/Hopper → no extra flags). No form params needed.
# Backward-compat: H3_USE_SAGE_ATTENTION is True iff ATTENTION is sage or sage+sol.
_builtins.H3_USE_SAGE_ATTENTION = ATTENTION in ('sage', 'sage+sol')
# Sol-Attn is enabled iff ATTENTION is sol or sage+sol.
_builtins.H3_USE_SOL_ATTENTION = ATTENTION in ('sol', 'sage+sol')
# kjnodes MiniMaxLowVRAMAttention (T4 low-VRAM path) is enabled iff
# ATTENTION is t4_lowmem. Mutually exclusive with sage/sol.
_builtins.H3_USE_T4_LOWMEM = ATTENTION == 't4_lowmem'
# Default steps per mode. The numerical values are from the jo-nike
# h3-turbo-eval grid.
#   - standard, fp8_dense → 20 / 16 (cleaner fp8 numerics at 16)
#   - all spectrum paths → 16 (Spectrum's adaptive schedule makes
#     step count less critical; the eval uses 20 with the Spectrum
#     columns getting ~6 forecasts out of 20 actual steps)
#   - all turbo paths (without spectrum) → 10 (LoRA sweet spot)
#   - turbo_lora_sage (without spectrum) → 8
#   - spectrum+turbo_lora → 12 (Spectrum lets the EMA LoRA stretch a
#     little since the trajectory is more accurate)
#   - spectrum+turbo_lora_sage → 10 (the eval's "emck8-sage-spec" column)
#   - lightx2v_4step → 4 (the upstream-recommended inference-steps)
#   - lightx2v_8step → 8 (the eval's "lx8-house" sweet spot; what
#     people are migrating to from 4-step since the trajectory is
#     cleaner and the visual quality is much better)
#   - lightx2v_*_ersde → +1 step vs the res_multistep version
#     (er_sde is slightly noisier per-step so 5/9 is the eval-tested
#     sweet spot for strength=0.75)
_MODE_DEFAULT_STEPS = {
    'standard': 20,
    'turbo_lora': 10,
    'turbo_lora_sage': 8,
    'spectrum': 16,
    'spectrum+turbo_lora': 12,
    'spectrum+turbo_lora_sage': 10,
    'fp8_dense': 16,
    'fp8_dense+turbo_lora': 10,
    'fp8_dense+turbo_lora_sage': 8,
    'fp8_dense+spectrum': 16,
    'fp8_dense+spectrum+turbo_lora': 12,
    'fp8_dense+spectrum+turbo_lora_sage': 10,
    'lightx2v_4step': 4,
    'lightx2v_8step': 8,
    'lightx2v_4step_ersde': 5,
    'lightx2v_8step_ersde': 9,
    'spectrum+lightx2v_4step': 6,
    'spectrum+lightx2v_8step': 10,
    'spectrum+lightx2v_4step_ersde': 7,
    'spectrum+lightx2v_8step_ersde': 11,
    'fp8_dense+lightx2v_4step': 4,
    'fp8_dense+lightx2v_8step': 8,
    'fp8_dense+lightx2v_4step_ersde': 5,
    'fp8_dense+lightx2v_8step_ersde': 9,
    'fp8_dense+spectrum+lightx2v_4step': 6,
    'fp8_dense+spectrum+lightx2v_8step': 10,
    'fp8_dense+spectrum+lightx2v_4step_ersde': 7,
    'fp8_dense+spectrum+lightx2v_8step_ersde': 11,
}[MODE]
# Adjust the turbo-path defaults if the user picked the Larryvrh
# backend. The Larryvrh Turbo node + v4-step600-EMA LoRA is documented
# at 6-8 steps (the upstream reference workflow ships 6 steps); the
# drbaph EMA ckpt500 LoRA + LoraLoaderModelOnly + SigmaShift path runs
# at 8-10 steps. Both share the same `turbo_lora*` mode name, so we
# pick the right default at runtime based on H3_TURBO_BACKEND.
if MODE in ('turbo_lora', 'turbo_lora_sage',
            'spectrum+turbo_lora', 'spectrum+turbo_lora_sage',
            'fp8_dense+turbo_lora', 'fp8_dense+turbo_lora_sage',
            'fp8_dense+spectrum+turbo_lora', 'fp8_dense+spectrum+turbo_lora_sage'):
    if getattr(__import__('builtins'), 'H3_TURBO_BACKEND', 'drbaph') == 'larryvrh':
        _MODE_DEFAULT_STEPS = {'turbo_lora': 6, 'turbo_lora_sage': 6,
                               'spectrum+turbo_lora': 8, 'spectrum+turbo_lora_sage': 8,
                               'fp8_dense+turbo_lora': 6, 'fp8_dense+turbo_lora_sage': 6,
                               'fp8_dense+spectrum+turbo_lora': 8,
                               'fp8_dense+spectrum+turbo_lora_sage': 8,
                              }[MODE]
print(f'  Mode         : {MODE}  (default steps: {_MODE_DEFAULT_STEPS}, sampler: {SAMPLER})')


### STEP 2｜下載模型權重
**r2v 使用者一定要把 `ENABLE_REF2VID` 保持為 True／勾選。** 會額外下載參考模式模型；不要中途關掉執行階段。

In [ ]:
#@title STEP 2｜下載模型（r2v 勾選 ENABLE_REF2VID）

import os, time, shutil
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi

# Comfy-Org/MiniMax-H3 layout (4 weights, ~40.8 GB on disk).
#   diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors  (19.5 GB)
#   text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors          (15.7 GB)
#   vae/minimax_h3_video_vae_fp16.safetensors                            ( 5.0 GB)
#   vae/minimax_h3_audio_vae_fp32.safetensors                            ( 0.6 GB)
# Total: ~40.8 GB on disk. The Drive cache keeps them across Colab sessions.
#
# Drive FUSE rejects symlinks and stalls on hardlinks of multi-GB files, so we
# full-copy into ComfyUI's models/ subdirs. We skip the copy when the dst already
# has the exact size from the Comfy-Org manifest (no spurious work on re-runs),
# and we verify sizes after the copy so a corrupted partial file gets caught
# here instead of as a `RuntimeError: shape '[...]' is invalid` deep in
# `comfy/utils.py load_safetensors` at sample time.

WEIGHTS_DIR = Path('/content/drive/MyDrive/AI_Video/AEI_3D_Cache/H3_ComfyUI/weights')
COMFY_DIR = Path('/content/drive/MyDrive/AI_Video/AEI_ComfyUI')

ENABLE_REF2VID = True  #@param {type:'boolean'}
# @markdown *Also download `minimax_h3_ref2va_pruned_int8_convrot.safetensors` (19.5 GB) for Reference-to-Video generation.*

# Pull MODE from STEP1's stored builtin (set in STEP1 as H3_MODE).
_MODE = getattr(__import__('builtins'), 'H3_MODE', 'standard')

PATTERNS = [
    'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',
    'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
    'vae/minimax_h3_video_vae_fp16.safetensors',
    'vae/minimax_h3_audio_vae_fp32.safetensors',
]
if ENABLE_REF2VID:
    PATTERNS.append('diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors')
# fp8_dense mode swaps the int8_convrot diffusion for the fp8_scaled one
# (added to Comfy-Org/MiniMax-H3 on 2025-08-04). Same 19.5 GB on disk, but
# the kernel runs fp8 matmuls natively so each sampler step is faster and
# the numerics at low step counts are noticeably cleaner. The on-disk
# byte count is identical to int8_convrot (19.5 GB), so Drive quota usage
# is unchanged. Compose with spectrum / turbo by prefixing the mode
# (e.g. `fp8_dense+spectrum+turbo_lora_sage`).
DIFF_PATTERN = ('diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors'
                if _MODE.startswith('fp8_') else
                'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors')
# Replace the int8 pattern with the fp8 one (they're mutually exclusive
# in PATTERNS — we only ever pull one diffusion file).
PATTERNS = [DIFF_PATTERN if p == 'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors' else p
            for p in PATTERNS]
# Turbo LoRA — drbaph's 4-step pruned checkpoint (~620 MB on disk, Apache 2.0).
# Combined with the Comfy-Org int8 base, this lets the sampler drop to
# 8-12 steps (~5x faster) at the cost of some sampling stability.
# The LoRA is loaded by LoraLoaderModelOnly at strength 1.0.
# Note: we use the EMA ckpt500 variant (minimax_h3_turbo_4step_ema_ckpt500_pruned_comfyui),
# not the non-EMA ckpt0. The EMA version is the "emck8" config from
# jo-nike's h3-turbo-eval and is the better one for sustained vocals and
# polyphony (the only test scene where the non-EMA still wins is the most
# aggressive short-clip cuts — we trade a little there for a lot elsewhere).
# Same ~700 MB on disk, same Apache 2.0 license, same drbaph repo.
TURBO_LORA = 'minimax_h3_turbo_4step_ema_ckpt500_pruned_comfyui.safetensors'
TURBO_LORA_PATH = 'loras/' + TURBO_LORA

# Alternative turbo backend: Larryvrh v4-step600-EMA LoRA, applied via
# the Larryvrh custom node (Larryvrh/ComfyUI-MiniMax-H3-Turbo) which
# drops in MiniMaxH3TurboLoRA + MiniMaxH3TurboSampler and auto-detects
# our pruned/int8 base to re-inject the LoRA's adaln update at run
# time. The upstream reference workflow pairs this LoRA with the Turbo
# Sampler at 6 steps; we use the same defaults. Slightly different
# distillation lineage than drbaph (CausVid-style vs drbaph GAN-style)
# and the v4-step600 EMA is the eval-and-author-recommended variant.
# Activated by STEP 1's TURBO_BACKEND = 'larryvrh'. Same Apache 2.0.
LARryvrh_LORA = 'minimax_h3_turbo_v4_step600_ema.safetensors'
LARryvrh_LORA_PATH = 'loras/' + LARryvrh_LORA
_TURBO_BACKEND = getattr(__import__('builtins'), 'H3_TURBO_BACKEND', 'drbaph')
if _TURBO_BACKEND == 'larryvrh':
    # Use the Larryvrh LoRA in place of the drbaph one. The workflow
    # workflow builder in STEP 5 switches to MiniMaxH3TurboLoRA + MiniMaxH3TurboSampler
    # based on this same H3_TURBO_BACKEND flag.
    TURBO_LORA = LARryvrh_LORA
    TURBO_LORA_PATH = LARryvrh_LORA_PATH

# lightx2v distillation LoRAs. As of 2026-08-09 lightx2v released v1.0
# (Apache 2.0; the v0.1 preview we used to ship is superseded). Two
# flavors live at lightx2v/Minimax-h3-Turbo (repo root, no `loras/`
# subdir — same layout as drbaph):
#
#   v1.0 8-step ComfyUI  (1.96 GB): trained 544p, σ shift 12/3, can run
#                                  at NFE=4 (slightly lower quality than
#                                  the dedicated 4-step) OR NFE=8 (eval's
#                                  "lx8-house" sweet spot). This is the
#                                  replacement for Kijai's
#                                  minimax_h3_fl2v_lightx2v_turbo_4step_v0.1.
#   v1.0 4-step 768p ComfyUI (1.96 GB): trained 1344x768, σ shift 6/3,
#                                  LoRA strength 1.0 (matches upstream's
#                                  --lora-alpha 128). Different shift from
#                                  544p variants — using 12/6 here is wrong.
# Both files load via plain LoraLoaderModelOnly (no custom node).
LIGHTX2V_LORA = 'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'
LIGHTX2V_LORA_768P = 'minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors'
LIGHTX2V_LORA_REPO = 'lightx2v/Minimax-h3-Turbo'
# Local destinations (under models/loras/) — same convention as drbaph
# (files at repo root, fetched directly into the loras/ subdir).
LIGHTX2V_LORA_PATH = 'loras/' + LIGHTX2V_LORA
LIGHTX2V_LORA_768P_PATH = 'loras/' + LIGHTX2V_LORA_768P

# The turbo half of the mode (after any fp8_dense+ or spectrum+ prefix).
# The turbo half of the mode is the rightmost `+` segment.
_TURBO_MODE = _MODE.rsplit('+', 1)[-1] if '+' in _MODE else _MODE
# The spectrum half is true when any of the segments equals 'spectrum'.
# (The fp8 prefix and turbo suffix can co-exist; spectrum is the middle
# one in the canonical order fp8_dense+spectrum+turbo_*, but we don't
# rely on positioning here — any segment equal to 'spectrum' triggers it.)
_SPECTRUM_MODE = 'spectrum' in _MODE.split('+')
# lightx2v segment detection: any rightmost segment starting with
# 'lightx2v_' is a lightx2v mode. The `_768p` suffix picks the v1.0
# 768p-trained LoRA (different file, different σ shift). All other
# lightx2v_* modes use the v1.0 8-step ComfyUI bf16 LoRA at the
# upstream 544p-trained resolution with σ shift 12/3 (ersde) or 12/6
# (house). Both files load via plain LoraLoaderModelOnly.
_LIGHTX2V_MODE = _TURBO_MODE.startswith('lightx2v_')
_LIGHTX2V_768P = _TURBO_MODE == 'lightx2v_4step_768p'
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    PATTERNS.append(TURBO_LORA_PATH)
if _LIGHTX2V_768P:
    PATTERNS.append(LIGHTX2V_LORA_768P_PATH)
elif _LIGHTX2V_MODE:
    PATTERNS.append(LIGHTX2V_LORA_PATH)

# Resolve expected sizes from the Hub manifest (one HTTP call, ~1s).
print('  Resolving expected sizes from Comfy-Org/MiniMax-H3 ...')
info = HfApi().repo_info('Comfy-Org/MiniMax-H3', files_metadata=True)
EXPECTED_BYTES = {sib.rfilename: sib.size for sib in info.siblings if sib.size is not None}

# When turbo modes are selected we also need the drbaph LoRA from a different
# repo. Merge its expected size so the verify step can validate the bytes,
# and we'll snapshot it separately below.
# When lightx2v modes are selected we also need the lightx2v LoRA from
# Kijai/MiniMax-H3_comfy. Same pattern: manifest + separate snapshot.
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    # Pull the LoRA's expected size from its own repo. Backend-dependent:
    #   - drbaph:    drbaph/MiniMax-H3-Turbo-Lora-ComfyUI (EMA ckpt500)
    #   - larryvrh:  larryvrh/MiniMax-H3-Turbo-LoRA (v4 step600 EMA)
    # Both files live at the REPO ROOT in their respective repos (no
    # loras/ subdir), and we download into WEIGHTS_DIR/loras/ directly
    # (see the snapshot_download call below).
    if _TURBO_BACKEND == 'larryvrh':
        print('  Resolving Turbo LoRA size from larryvrh/MiniMax-H3-Turbo-LoRA ...')
        _turbo_repo_id = 'larryvrh/MiniMax-H3-Turbo-LoRA'
    else:
        print('  Resolving Turbo LoRA size from drbaph/MiniMax-H3-Turbo-Lora-ComfyUI ...')
        _turbo_repo_id = 'drbaph/MiniMax-H3-Turbo-Lora-ComfyUI'
    _tb_info = HfApi().repo_info(_turbo_repo_id, files_metadata=True)
    _tb_size = None
    for sib in _tb_info.siblings:
        if sib.rfilename == TURBO_LORA and sib.size is not None:
            _tb_size = sib.size
            break
    if _tb_size is None:
        # Older huggingface_hub versions (or repos where HF doesn't expose
        # blob metadata through files_metadata) leave sib.size as None.
        # Fall back to a HEAD request on the resolve URL, which always
        # returns the actual content length.
        print(f'  size missing in HF manifest; HEAD-resolving {_turbo_repo_id}/{TURBO_LORA} ...')
        _req = urllib.request.Request(
            f'https://huggingface.co/{_turbo_repo_id}/resolve/main/{TURBO_LORA}',
            method='HEAD')
        with urllib.request.urlopen(_req, timeout=30) as _r:
            _tb_size = int(_r.headers['content-length'])
    EXPECTED_BYTES[TURBO_LORA_PATH] = _tb_size
if _LIGHTX2V_MODE:
    # Resolve sizes for whichever lightx2v LoRA(s) this mode needs from
    # the upstream lightx2v/Minimax-h3-Turbo repo. Both files live at
    # the repo root (no `loras/` subdir) and ship as ComfyUI bf16.
    print(f'  Resolving lightx2v LoRA size(s) from {LIGHTX2V_LORA_REPO} ...')
    _lx_info = HfApi().repo_info(LIGHTX2V_LORA_REPO, files_metadata=True)
    _lx_targets = []
    if _LIGHTX2V_768P:
        _lx_targets.append((LIGHTX2V_LORA_768P, LIGHTX2V_LORA_768P_PATH))
    else:
        _lx_targets.append((LIGHTX2V_LORA, LIGHTX2V_LORA_PATH))
    for _lx_filename, _lx_path in _lx_targets:
        _lx_size = None
        for sib in _lx_info.siblings:
            if sib.rfilename == _lx_filename and sib.size is not None:
                _lx_size = sib.size
                break
        if _lx_size is None:
            print(f'  size missing in HF manifest; HEAD-resolving '
                  f'{LIGHTX2V_LORA_REPO}/{_lx_filename} ...')
            _req = urllib.request.Request(
                f'https://huggingface.co/{LIGHTX2V_LORA_REPO}/resolve/main/{_lx_filename}',
                method='HEAD')
            with urllib.request.urlopen(_req, timeout=30) as _r:
                _lx_size = int(_r.headers['content-length'])
        EXPECTED_BYTES[_lx_path] = _lx_size

t0 = time.time()
weights = snapshot_download(
    repo_id='Comfy-Org/MiniMax-H3',
    local_dir=str(WEIGHTS_DIR),
    allow_patterns=PATTERNS,
)
print(f'  Weights cached at {weights} in {time.time() - t0:.0f}s')

# Pull the Turbo LoRA from its own repo if it isn't already in WEIGHTS_DIR
# (snapshot_download above only matched Comfy-Org patterns).
# Note: drbaph's files live at the REPO ROOT (no `loras/` subdir), unlike
# Kijai/MiniMax-H3_comfy which puts files under loras/. We download with
# a bare-filename allow_pattern into WEIGHTS_DIR/loras/ directly so the
# file lands at the expected destination without an extra move step.
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    _lora_dst = WEIGHTS_DIR / TURBO_LORA_PATH
    _lora_dir_for_fetch = WEIGHTS_DIR / 'loras'
    _lora_dir_for_fetch.mkdir(parents=True, exist_ok=True)
    if not _lora_dst.exists() or _lora_dst.stat().st_size != EXPECTED_BYTES.get(TURBO_LORA_PATH, -1):
        _turbo_repo = ('larryvrh/MiniMax-H3-Turbo-LoRA'
                       if _TURBO_BACKEND == 'larryvrh' else
                       'drbaph/MiniMax-H3-Turbo-Lora-ComfyUI')
        print(f'  Pulling Turbo LoRA {TURBO_LORA} from {_turbo_repo} ...')
        snapshot_download(
            repo_id=_turbo_repo,
            local_dir=str(_lora_dir_for_fetch),
            allow_patterns=[TURBO_LORA],
        )

# Pull the lightx2v LoRA(s) from the upstream lightx2v/Minimax-h3-Turbo
# repo if they aren't already in WEIGHTS_DIR. Same pattern as drbaph:
# files live at the repo root (no `loras/` subdir), so we download with
# a bare-filename allow_pattern into WEIGHTS_DIR/loras/ directly.
if _LIGHTX2V_MODE:
    _lora_dir_for_fetch = WEIGHTS_DIR / 'loras'
    _lora_dir_for_fetch.mkdir(parents=True, exist_ok=True)
    if _LIGHTX2V_768P:
        _lx_fetches = [(LIGHTX2V_LORA_768P, LIGHTX2V_LORA_768P_PATH)]
    else:
        _lx_fetches = [(LIGHTX2V_LORA, LIGHTX2V_LORA_PATH)]
    for _lx_filename, _lx_path in _lx_fetches:
        _lx_dst = WEIGHTS_DIR / _lx_path
        if not _lx_dst.exists() or _lx_dst.stat().st_size != EXPECTED_BYTES.get(_lx_path, -1):
            print(f'  Pulling lightx2v LoRA {_lx_filename} from {LIGHTX2V_LORA_REPO} ...')
            snapshot_download(
                repo_id=LIGHTX2V_LORA_REPO,
                local_dir=str(_lora_dir_for_fetch),
                allow_patterns=[_lx_filename],
            )

DIFF = COMFY_DIR / 'models' / 'diffusion_models'
TXT  = COMFY_DIR / 'models' / 'text_encoders'
VAE  = COMFY_DIR / 'models' / 'vae'
for d in (DIFF, TXT, VAE):
    d.mkdir(parents=True, exist_ok=True)

def _expected_bytes(path_rel: str) -> int:
    return EXPECTED_BYTES.get(path_rel, -1)


def _wire(src: Path, dst: Path, expected_size: int):
    """Copy `src` to `dst` unless `dst` already matches `expected_size`.

    Skips the copy when the existing dst size matches the manifest exactly
    (cheap ~ms check, saves 8-25 min of Drive FUSE copy on re-runs).
    Re-runs that have a partial dst fall through to the full copy.
    Drives a 5-second heartbeat so the user sees progress.
    """
    expected_str = f'{expected_size/1024**3:.2f} GB'
    if dst.exists():
        existing = dst.stat().st_size
        if existing == expected_size:
            print(f'    OK  {dst.name}  ({expected_str} verified)', flush=True)
            return
        # Wrong size: usually a partial write from an interrupted earlier run.
        print(f'    -> {dst.name}: dst is {existing/1024**3:.2f} GB but expected {expected_str}; replacing.', flush=True)
        dst.unlink()
    elif expected_size <= 0:
        print(f'    ?? {dst.name}: no manifest size available; copying without verification.', flush=True)

    src_size_gb = src.stat().st_size / 1024**3
    print(f'    -> {dst.name}  ({src_size_gb:.1f} GB) ...', flush=True)
    t0 = time.time()
    copied = 0
    last_beat = time.time()
    with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
        while True:
            chunk = fsrc.read(64 * 1024 * 1024)  # 64 MB chunks
            if not chunk:
                break
            fdst.write(chunk)
            copied += len(chunk)
            if time.time() - last_beat > 5:
                pct = 100 * copied / (src_size_gb * 1024**3)
                rate = copied / (time.time() - t0) / 1024**2
                print(f'       {pct:5.1f}%  {rate:.0f} MB/s', flush=True)
                last_beat = time.time()
    print(f'    -> done in {time.time() - t0:.0f}s', flush=True)


def _verify():
    """Final sanity check: every dst matches the manifest size exactly.
    If anything is off, raise with a delete-and-retry message."""
    bad = []
    for pat in PATTERNS:
        dst = COMFY_DIR / 'models' / pat
        exp = _expected_bytes(pat)
        if not dst.exists():
            bad.append((pat, 0, exp))
            continue
        got = dst.stat().st_size
        if got != exp:
            bad.append((pat, got, exp))
    if bad:
        msg = 'Some weights do not match the Comfy-Org manifest.\n'
        for pat, got, exp in bad:
            msg += f'  {pat}: got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB\n'
        msg += ('Delete the bad files (e.g. '
                '`shutil.rmtree(\'/content/drive/MyDrive/AI_Video/AEI_ComfyUI/models\')`) and re-run STEP 2.')
        raise SystemExit(msg)


_wire(WEIGHTS_DIR / DIFF_PATTERN, DIFF / Path(DIFF_PATTERN).name,
         _expected_bytes(DIFF_PATTERN))
if ENABLE_REF2VID:
    _r2v_pat = 'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors'
    _wire(WEIGHTS_DIR / _r2v_pat, DIFF / 'minimax_h3_ref2va_pruned_int8_convrot.safetensors',
          _expected_bytes(_r2v_pat))
_wire(WEIGHTS_DIR / 'text_encoders' / 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
         TXT / 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
         _expected_bytes('text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'))
_wire(WEIGHTS_DIR / 'vae' / 'minimax_h3_video_vae_fp16.safetensors',
         VAE / 'minimax_h3_video_vae_fp16.safetensors',
         _expected_bytes('vae/minimax_h3_video_vae_fp16.safetensors'))
_wire(WEIGHTS_DIR / 'vae' / 'minimax_h3_audio_vae_fp32.safetensors',
         VAE / 'minimax_h3_audio_vae_fp32.safetensors',
         _expected_bytes('vae/minimax_h3_audio_vae_fp32.safetensors'))
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage') or _LIGHTX2V_MODE:
    LORA_DIR = COMFY_DIR / 'models' / 'loras'
    LORA_DIR.mkdir(parents=True, exist_ok=True)
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    _wire(WEIGHTS_DIR / TURBO_LORA_PATH, LORA_DIR / TURBO_LORA,
         _expected_bytes(TURBO_LORA_PATH))
if _LIGHTX2V_MODE:
    _wire(WEIGHTS_DIR / LIGHTX2V_LORA_PATH, LORA_DIR / LIGHTX2V_LORA,
         _expected_bytes(LIGHTX2V_LORA_PATH))

_verify()

print('  Weights wired into ComfyUI/models/{diffusion_models,text_encoders,vae}/ (sizes verified against Comfy-Org/MiniMax-H3 manifest).')
if _MODE.startswith('fp8_'):
    print(f'  fp8_dense: diffusion model is the fp8_scaled checkpoint (no INT8 dequant at sample time).')
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    print(f'  Turbo (drbaph EMA-ckpt500) LoRA wired into ComfyUI/models/loras/ — strength 1.0 at denoise time.')
if _LIGHTX2V_MODE:
    _lx_strength = 0.75 if _TURBO_MODE.endswith('_ersde') else 1.0
    print(f'  lightx2v 4-step LoRA wired into ComfyUI/models/loras/ — strength {_lx_strength} at denoise time.')
total = 0
for d in (DIFF, TXT, VAE):
    for p in sorted(d.iterdir()):
        if p.suffix == '.safetensors':
            sz = p.stat().st_size / 1024**3
            total += sz
            print(f'    {sz:5.2f} GB  {d.name}/{p.name}')
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage') or _LIGHTX2V_MODE:
    for p in sorted(LORA_DIR.iterdir()):
        if p.suffix == '.safetensors':
            sz = p.stat().st_size / 1024**3
            total += sz
            print(f'    {sz:5.2f} GB  loras/{p.name}')
print(f'  Total on ComfyUI side: {total:.2f} GB')


### STEP 3｜啟動 ComfyUI 背景服務
按 ▶ 啟動 ComfyUI。這格不會顯示 Gradio；完成後請執行 STEP 5 初始化工作流程。


In [ ]:
#@title STEP 3｜啟動 ComfyUI

import os, sys, time, signal, subprocess, urllib.request, urllib.error, json, importlib
from pathlib import Path

COMFY_DIR = Path('/content/drive/MyDrive/AI_Video/AEI_ComfyUI')
COMFY_HOST = '127.0.0.1'
COMFY_PORT = 8188
COMFY_URL  = f'http://{COMFY_HOST}:{COMFY_PORT}'

# Pinned memory is the entire reason this notebook exists: the default ComfyUI policy pins
# ~90% of host RAM. With 53 GB of RAM the pinned ceiling is ~47 GB, and the diffusers runtime
# OOM-kills long before the model fits. --disable-pinned-memory is the single flag that makes
# this recipe work on 24-32 GB RAM (see tonyd2wild/minimax-h3-local).
# --fp16-intermediates halves inter-node tensors and is cheap insurance.
LAUNCH_CMD = [
    sys.executable, 'main.py',
    '--listen', COMFY_HOST,
    '--port', str(COMFY_PORT),
    '--disable-pinned-memory',
    '--fp16-intermediates',
    '--disable-api-nodes',
    '--output-directory', str(COMFY_DIR / 'output'),
    '--input-directory', str(COMFY_DIR / 'input'),
]
# Auto-detect sub-16GB-VRAM hosts (Turing sm_75) and enable the launch
# flags that materially change the tradeoff on those cards. On roomy GPUs
# (Ampere/Ada/Hopper, sm_80+) the model fits in VRAM and dynamic-VRAM
# offload doesn't trigger, so these flags are no-ops.
#
#   --reserve-vram 0.9 : keep 0.9 GB free for the OS / other processes
#   --fast-disk        : put the dynamic-VRAM offload cache on the
#                        NVMe-backed /content drive (faster than host RAM
#                        swaps). Safe on L4+ too but does no work.
#
# The old form params (RESERVE_VRAM_GB / FAST_DISK / USE_SAGE_ATTENTION)
# were removed in commit 82119de but the references to them in this cell
# were not cleaned up, so STEP 1 crashed with NameError on every launch.
# Auto-detection makes the flags GPU-appropriate without exposing them
# in STEP 1.
_GPU_CC_AUTO = None
if _torch_check.cuda.is_available():
    _gp_auto = _torch_check.cuda.get_device_properties(0)
    _GPU_CC_AUTO = float(f'{_gp_auto.major}.{_gp_auto.minor}')
    if _GPU_CC_AUTO < 8.0:
        # Turing (sm_75) — T4 / T40 / V100-not-really-V100. CPU-offload
        # bound; needs both flags.
        LAUNCH_CMD.extend(['--reserve-vram', '0.9'])
        LAUNCH_CMD.append('--fast-disk')
        print(f'  GPU compute capability {_GPU_CC_AUTO} (Turing) detected — '
              'auto-enabled --reserve-vram 0.9 --fast-disk.')
    else:
        print(f'  GPU compute capability {_GPU_CC_AUTO} (Ampere/Ada/Hopper) — '
              'no extra launch flags needed.')
env = os.environ.copy()

# Kill any leftover ComfyUI subprocess before launching a fresh one.
# Two patterns to be defensive.
#   1. pkill by command-line substring (matches 'python ... AEI_ComfyUI/main.py')
#   2. fuser on the port itself (kills anything bound to TCP 8188 even
#      if the command line doesn't match the substring above, e.g. a
#      ComfyUI launched from a different path or via a wrapper script)
# Without this, a previous Colab session's ComfyUI can hold port 8188
# and the new launch fails silently with 'Port 8188 is already in use'
# while STEP 3's readiness check still passes (because the OLD server
# is responding to /system_stats). STEP 7 then polls /history forever
# because the prompt_id was queued against the new (failed) server but
# the polling hits the old (responding) one.
subprocess.run(['pkill', '-9', '-f', f'{COMFY_DIR.name}.*main.py'], check=False)
subprocess.run(['pkill', '-9', '-f', 'AEI_ComfyUI'], check=False)
time.sleep(2)
# fuser is in psmisc; on Colab it's sometimes missing, so we wrap in try.
for _cmd in (['fuser', '-k', '8188/tcp'], ['lsof', '-ti', ':8188']):
    try:
        r = subprocess.run(_cmd, capture_output=True, timeout=5, check=False)
        # fuser prints "8188/tcp: <pid>..." to stderr; lsof prints pids to stdout.
        # We don't fail the cell if neither command exists — pkill above already
        # handled the common case.
        if r.returncode == 0 and (r.stdout or r.stderr):
            time.sleep(2)
            print(f'  Killed leftover process(es) on port 8188 via {chr(32).join(_cmd)}')
            break
    except (FileNotFoundError, subprocess.TimeoutExpired):
        continue

(COMFY_DIR / 'output').mkdir(parents=True, exist_ok=True)
(COMFY_DIR / 'input').mkdir(parents=True, exist_ok=True)

# Optional: install kjnodes custom node if MODE = turbo_lora_sage. kjnodes
# provides the MiniMaxH3MemoryEfficientSageAttentionPatch node that drops
# attention memory ~2x. We skip the install for other modes since the
# 620 MB of upstream deps would be wasted.
# Optional: install ComfyUI-Spectrum-MiniMax-H3 if the mode contains the
# `spectrum` segment. This is xmarre's Chebyshev-ridge feature-forecaster
# (https://github.com/xmarre/ComfyUI-Spectrum-MiniMax-H3) and ships
# SpectrumApplyMiniMaxH3. No Python deps of its own (uses only torch +
# ComfyUI modules), so the only cost is ~2 MB of source + a git clone.
_MODE3_pre = getattr(__import__('builtins'), 'H3_MODE', 'standard')
_TURBO_MODE3_pre = _MODE3_pre.rsplit('+', 1)[-1] if '+' in _MODE3_pre else _MODE3_pre
_SPECTRUM_MODE3_pre = 'spectrum' in _MODE3_pre.split('+')
_TURBO_BACKEND3 = getattr(__import__('builtins'), 'H3_TURBO_BACKEND', 'drbaph')
# Install ComfyUI-KJNodes whenever the active mode is turbo_lora_sage
# (Sage is the workflow path's Sage patch) OR the user picked 'sage',
# 'sage+sol', or 't4_lowmem' for the attention backend. The KJNodes
# package provides MiniMaxH3MemoryEfficientSageAttentionPatch (for
# `sage`/`sage+sol`) AND the newer MiniMaxLowVRAMAttention +
# MiniMaxChunkFeedForward nodes used by the `t4_lowmem` path (Turing
# T4 compatible — works on every arch including sm_75 where Sage/Sol
# refuse to compile).
_ATTN3 = getattr(__import__('builtins'), 'H3_ATTENTION', 'sage')
_NEEDS_KJ = _TURBO_MODE3_pre == 'turbo_lora_sage' or _ATTN3 in ('sage', 'sage+sol', 't4_lowmem')
if _NEEDS_KJ:
    _CUSTOM = COMFY_DIR / 'custom_nodes' / 'ComfyUI-KJNodes'
    if not _CUSTOM.exists():
        _kj_reason = ('SageAttention / H3 utilities' if _ATTN3 != 't4_lowmem'
                      else 'MiniMaxLowVRAMAttention (T4 low-VRAM path)')
        print(f'  Cloning ComfyUI-KJNodes (for {_kj_reason}) ...')
        subprocess.run(['git', 'clone', '--depth=1',
                        'https://github.com/kijai/ComfyUI-KJNodes.git', str(_CUSTOM)], check=True)
        req = _CUSTOM / 'requirements.txt'
        if req.exists():
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(req)], check=False)
    else:
        print(f'  Reusing existing {_CUSTOM}')
    # The kjnodes MiniMaxH3MemoryEfficientSageAttentionPatch imports
    # `sageattention.core.get_cuda_arch_versions` (from the thu-ml
    # SageAttention package, NOT the `sageattn` wrapper). If the pip
    # package is missing or too old, the node crashes with
    # "sageattention is not new enough version or could not determine
    # CUDA architecture" the first time the workflow runs (caught on
    # L4 run with ATTENTION=sage+sol). The kjnodes repo's
    # requirements.txt only pulls the wrapper `sageattn`, not the
    # underlying `sageattention`; install it here for any mode that
    # needs the Sage patch (sage / sage+sol, also the implicit
    # turbo_lora_sage path).
    #
    # IMPORTANT: pip install sageattention from PyPI gives 1.0.6 (the
    # last tagged release) which is MISSING get_cuda_arch_versions —
    # the kjnodes H3 Sage patch needs that helper. Install from GitHub
    # main instead, where get_cuda_arch_versions is present. The build
    # process compiles CUDA kernels via torch.utils.cpp_extension JIT —
    # takes 5-15 min on first run (no prebuilt wheels exist on PyPI or
    # in any release asset). The compiled extension is cached at
    # ~/.cache/torch_extensions/<py_version>/<arch>/, so subsequent
    # runs in the same session are ~10 seconds.
    #
    # Cross-session persistence: we copy the torch_extensions cache to
    # the Drive weight cache after a successful build (path below) and
    # restore from there on the next session, skipping the build
    # entirely if the cached wheel is still compatible.
    if _ATTN3 in ('sage', 'sage+sol') or _TURBO_MODE3_pre == 'turbo_lora_sage':
        _SAGE_EXT_CACHE = Path('/content/drive/MyDrive/AI_Video/AEI_3D_Cache/sage_extensions')
        # 1. Restore from Drive cache if present.
        import shutil as _shutil
        _LOCAL_TORCH_EXT = Path('/root/.cache/torch_extensions')
        if _SAGE_EXT_CACHE.exists() and not _LOCAL_TORCH_EXT.exists():
            try:
                _LOCAL_TORCH_EXT.parent.mkdir(parents=True, exist_ok=True)
                _shutil.copytree(_SAGE_EXT_CACHE, _LOCAL_TORCH_EXT, dirs_exist_ok=True)
                print(f'  Restored sage extensions cache from Drive ({_SAGE_EXT_CACHE}).')
            except Exception as _e:
                print(f'  Could not restore sage extensions cache: {_e}')
        _HAS_SAGE_PKG = False
        try:
            import importlib
            import sageattention  # noqa: F401
            if 'sageattention' in sys.modules:
                importlib.reload(sys.modules['sageattention'])
            from sageattention.core import get_cuda_arch_versions  # noqa: F401
            _HAS_SAGE_PKG = True
        except (ImportError, AttributeError):
            pass
        if not _HAS_SAGE_PKG:
            print('  ⚠ thu-ml sageattention not installed (or too old — '
                  'PyPI 1.0.6 is missing get_cuda_arch_versions).')
            print('    Installing from GitHub main (compiles CUDA wheel, 5-15 min on L4)...')
            print('    (If install keeps failing, run this manually in a Colab cell:')
            print('     !pip install --no-build-isolation git+https://github.com/thu-ml/SageAttention@main')
            print('     then re-run STEP 3. As a fallback, switch ATTENTION to sol.)')
            # Make sure the build toolchain is present. Colab fresh
            # envs often lack git / ninja / nvcc on PATH; torch's
            # cpp_extension needs nvcc to compile the kernels.
            #
            # CRITICAL: SageAttention's setup.py checks `nvcc --version`
            # against compute capabilities. For sm_89 (L4/A4000/Ada
            # Lovelace) it requires CUDA >=12.4; for sm_90 (H100) it
            # requires >=12.3. The default Colab runtime's /usr/local/cuda
            # has nvcc 12.3 or older (predates the 2025 SageAttention
            # build requirements), so the build dies at the
            # "CUDA 12.4 or higher is required for compute capability
            # 8.9" RuntimeError with the truncated "uild wheel did not
            # run successfully" message that hid the real cause.
            #
            # Fix: apt-install nvidia-cuda-toolkit which gives us a
            # newer nvcc (12.6+ on Colab 2026 base images). The pip
            # wheels (torch + CUDA libs) are unrelated to this — nvcc
            # only ships with the apt package, not the pip packages.
            try:
                _shutil_local = importlib.import_module('shutil').which
                if _shutil_local('git') is None:
                    print('    Installing git via apt...')
                    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git'],
                                   capture_output=True, timeout=120, check=False)
                if _shutil_local('ninja') is None:
                    print('    Installing ninja-build via pip...')
                    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                                    'ninja'],
                                   capture_output=True, timeout=60, check=False)
                # The big one: make sure nvcc is new enough. Check
                # current version; if <12.4, install nvidia-cuda-toolkit
                # (gives nvcc 12.6 on Colab 2026). Skip if already
                # recent.
                def _nvcc_version():
                    try:
                        r = subprocess.run(['nvcc', '--version'],
                                           capture_output=True, timeout=10)
                        out = r.stdout.decode(errors='replace')
                        # "release 12.4.0" → 12.4
                        import re as _re
                        m = _re.search(r'release\s+(\d+)\.(\d+)', out)
                        if m:
                            return float(f'{m.group(1)}.{m.group(2)}')
                    except Exception:
                        pass
                    return None
                _nvcc_v = _nvcc_version()
                if _nvcc_v is None or _nvcc_v < 12.4:
                    print(f'    nvcc version: {_nvcc_v or "(not found)"} — too old for sm_89/SageAttention.')
                    print('    Installing nvidia-cuda-toolkit (gives nvcc 12.6+)...')
                    _apt = subprocess.run(['apt-get', 'install', '-y', '-qq',
                                           'nvidia-cuda-toolkit'],
                                          capture_output=True, timeout=300, check=False)
                    if _apt.returncode == 0:
                        # After apt-install, nvcc lands at
                        # /usr/bin/nvcc (or /usr/local/cuda/bin/nvcc
                        # depending on the image). Make sure PATH sees
                        # the new one.
                        for _candidate in ('/usr/local/cuda/bin/nvcc',
                                           '/usr/bin/nvcc'):
                            if os.path.exists(_candidate):
                                os.environ['CUDA_HOME'] = os.path.dirname(
                                    os.path.dirname(_candidate))
                                os.environ['PATH'] = os.path.dirname(
                                    _candidate) + ':' + os.environ.get('PATH', '')
                                break
                        _nvcc_v = _nvcc_version()
                        print(f'    ✓ nvcc now at version: {_nvcc_v}')
                    else:
                        print(f'    ✗ nvidia-cuda-toolkit install failed (rc={_apt.returncode});')
                        print('      stderr:', _apt.stderr.decode(errors='replace')[-400:])
                else:
                    print(f'    nvcc version: {_nvcc_v} (>=12.4, OK).')
            except Exception as _e:
                print(f'    ⚠ toolchain pre-install failed (will let pip surface): {_e}')
            # Print nvcc availability for the user's debugging
            _nvcc = importlib.import_module('shutil').which('nvcc')
            if _nvcc is None:
                # Common Colab path: /usr/local/cuda/bin/nvcc exists
                # but isn't on PATH. Export CUDA_HOME so torch's
                # cpp_extension finds it.
                if os.path.exists('/usr/local/cuda/bin/nvcc'):
                    print('    nvcc not on PATH but found at /usr/local/cuda/bin/nvcc; exporting CUDA_HOME.')
                    os.environ['CUDA_HOME'] = '/usr/local/cuda'
                    os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ.get('PATH', '')
                else:
                    print('    nvcc not found at /usr/local/cuda/bin/nvcc either; install will fail.')
                    print('      Fix: !apt-get install -y -qq nvidia-cuda-toolkit')
            try:
                _install_result = subprocess.run(
                    [sys.executable, '-m', 'pip', 'install', '--quiet',
                     '--no-build-isolation',
                     'git+https://github.com/thu-ml/SageAttention@main'],
                    capture_output=True, timeout=1800, check=False)
                if _install_result.returncode != 0:
                    print('    ✗ pip install failed (rc={}); last 2000 chars:'.format(
                        _install_result.returncode))
                    print('      stdout:', _install_result.stdout.decode(errors='replace')[-1000:])
                    print('      stderr:', _install_result.stderr.decode(errors='replace')[-1000:])
                else:
                    print('    pip install returned 0; verifying import...')
                # Verify the import now succeeds with the version-gated API.
                try:
                    import importlib
                    if 'sageattention' in sys.modules:
                        importlib.reload(sys.modules['sageattention'])
                    import sageattention  # noqa: F401
                    from sageattention.core import get_cuda_arch_versions  # noqa: F401
                    print('    ✓ sageattention installed from GitHub main; kjnodes H3 Sage patch ready.')
                    # 2. Save the compiled extension cache to Drive for next session.
                    try:
                        if _LOCAL_TORCH_EXT.exists():
                            _SAGE_EXT_CACHE.mkdir(parents=True, exist_ok=True)
                            _shutil.copytree(_LOCAL_TORCH_EXT, _SAGE_EXT_CACHE, dirs_exist_ok=True)
                            print(f'    ✓ cached compiled extensions to {_SAGE_EXT_CACHE} for next session.')
                    except Exception as _e:
                        print(f'    ⚠ could not cache extensions: {_e}')
                except (ImportError, AttributeError) as _e:
                    print(f'    ✗ install completed but import still fails ({type(_e).__name__}: {_e});')
                    print('      Sage will not work — switch ATTENTION to "none" or "sol".')
            except subprocess.TimeoutExpired:
                print('    ✗ pip install sageattention timed out after 30 min;')
                print('      Sage will not work — switch ATTENTION to "none" or "sol".')
        else:
            print('  ✓ thu-ml sageattention already installed and current.')

# Install Saganaki22/ComfyUI-sol-attn when 'sol' or 'sage+sol' is
# picked. The repo ships `MiniMaxH3MemoryEfficientSolAttentionPatch`
# and `MiniMaxH3ScheduledSolAttentionPatch` for the H3 model. SM89+
# only (L4+); on Turing the kernel falls back to dense. No Python
# deps of its own (Triton only, which ComfyUI already has).
_VHS3 = COMFY_DIR / 'custom_nodes' / 'ComfyUI-VideoHelperSuite'
if not _VHS3.exists():
    print('  Cloning ComfyUI-VideoHelperSuite (video loading for Ref2Vid) ...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git', str(_VHS3)], check=True)

if _ATTN3 in ('sol', 'sage+sol'):
    _SOL = COMFY_DIR / 'custom_nodes' / 'ComfyUI-sol-attn'
    if not _SOL.exists():
        print('  Cloning ComfyUI-sol-attn (Sol-Attn + H3 fused modulation) ...')
        subprocess.run(['git', 'clone', '--depth=1',
                        'https://github.com/Saganaki22/ComfyUI-sol-attn.git', str(_SOL)], check=True)
    else:
        print(f'  Reusing existing {_SOL}')
# Optional: install Larryvrh Turbo node when TURBO_BACKEND=larryvrh. This
# node (https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo) provides
# MiniMaxH3TurboLoRA (run-time LoRA bypass) and MiniMaxH3TurboSampler
# (4-step audio/video dual-schedule). Bundled silu_t_emb_grid.safetensors
# (~13 MB) lets the node auto-detect our pruned/int8 base and re-inject
# the LoRA's adaln update at run time. No Python deps of its own.
if _TURBO_BACKEND3 == 'larryvrh' and _TURBO_MODE3_pre in ('turbo_lora', 'turbo_lora_sage'):
    _LARRYVRH = COMFY_DIR / 'custom_nodes' / 'ComfyUI-MiniMax-H3-Turbo'
    if not _LARRYVRH.exists():
        print('  Cloning ComfyUI-MiniMax-H3-Turbo (Larryvrh Turbo node) ...')
        subprocess.run(['git', 'clone', '--depth=1',
                        'https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git', str(_LARRYVRH)], check=True)
    else:
        print(f'  Reusing existing {_LARRYVRH}')
if _SPECTRUM_MODE3_pre:
    _SPECTRUM = COMFY_DIR / 'custom_nodes' / 'ComfyUI-Spectrum-MiniMax-H3'
    if not _SPECTRUM.exists():
        print('  Cloning ComfyUI-Spectrum-MiniMax-H3 (Spectral feature forecasting) ...')
        subprocess.run(['git', 'clone', '--depth=1',
                        'https://github.com/xmarre/ComfyUI-Spectrum-MiniMax-H3.git', str(_SPECTRUM)], check=True)
    else:
        print(f'  Reusing existing {_SPECTRUM}')

print(f'  Launching: {" ".join(LAUNCH_CMD)}')
log_path = COMFY_DIR / 'comfyui.log'
log_f = open(log_path, 'wb')
proc = subprocess.Popen(
    LAUNCH_CMD,
    cwd=str(COMFY_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setsid,
)
print(f'  PID: {proc.pid}, log: {log_path}')

ready = False
deadline = time.time() + 600
while time.time() < deadline:
    # If the new process we spawned has already exited (e.g. port conflict,
    # missing dependency, CUDA init failure), bail out before checking
    # /system_stats — that endpoint might still respond because some
    # leftover ComfyUI from a previous session is bound to 8188.
    if proc.poll() is not None:
        print(f'  ComfyUI exited early with code {proc.returncode}. Last 40 log lines:')
        with open(log_path) as f:
            tail = f.read().splitlines()[-40:]
            for ln in tail:
                print('   ', ln)
        raise SystemExit('ComfyUI failed to start.')
    try:
        with urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=2) as r:
            r.read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError, OSError):
        pass
    time.sleep(2)

if not ready:
    proc.terminate()
    raise SystemExit(f'ComfyUI did not respond within 10 minutes. Tail of log:\n'
                     + '\n'.join(open(log_path).read().splitlines()[-20:]))

# Sanity check the model files BEFORE we hand the workflow off. ComfyUI's
# load_torch_file reads each tensor with `view(info["shape"])`, so a partial
# file whose metadata header says 33.5M elements but whose bytes are only
# 11.9M will throw `RuntimeError: shape '[...]' is invalid` deep inside the
# load. Catching it here instead of at denoise time is the difference
# between "fast feedback" and "45 min of waiting for the sampler to fail".
from huggingface_hub import HfApi as _HfApi
_HF_API = _HfApi()
_INFO = _HF_API.repo_info('Comfy-Org/MiniMax-H3', files_metadata=True)
_EXPECTED = {sib.rfilename: sib.size for sib in _INFO.siblings if sib.size is not None}
_MODE3 = getattr(__import__('builtins'), 'H3_MODE', 'standard')
_TURBO_MODE3 = _MODE3.rsplit('+', 1)[-1] if '+' in _MODE3 else _MODE3
if _TURBO_MODE3 in ('turbo_lora', 'turbo_lora_sage'):
    # Pull the LoRA's expected size from its own repo. Backend-dependent.
    if _TURBO_BACKEND3 == 'larryvrh':
        _turbo_repo_id3 = 'larryvrh/MiniMax-H3-Turbo-LoRA'
    else:
        _turbo_repo_id3 = 'drbaph/MiniMax-H3-Turbo-Lora-ComfyUI'
    _TURBO_INFO = _HF_API.repo_info(_turbo_repo_id3, files_metadata=True)
    _tb3_size = None
    for sib in _TURBO_INFO.siblings:
        if sib.rfilename == TURBO_LORA and sib.size is not None:
            _tb3_size = sib.size
            break
    if _tb3_size is None:
        # HF manifest returned no size; fall back to HEAD.
        _req = urllib.request.Request(
            f'https://huggingface.co/{_turbo_repo_id3}/resolve/main/{TURBO_LORA}',
            method='HEAD')
        with urllib.request.urlopen(_req, timeout=30) as _r:
            _tb3_size = int(_r.headers['content-length'])
    _EXPECTED['loras/' + TURBO_LORA] = _tb3_size
# Pick the diffusion file based on MODE: fp8_scaled for fp8_* modes,
# int8_convrot for the rest. The text encoder + VAE patterns are unchanged.
_DIFF_PAT3 = ('diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors'
              if _MODE3.startswith('fp8_') else
              'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors')
_PATTERNS = [
    _DIFF_PAT3,
    'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
    'vae/minimax_h3_video_vae_fp16.safetensors',
    'vae/minimax_h3_audio_vae_fp32.safetensors',
]
if _TURBO_MODE3 in ('turbo_lora', 'turbo_lora_sage'):
    _PATTERNS.append('loras/' + TURBO_LORA)
print('  Verifying ComfyUI/models file sizes against Comfy-Org/MiniMax-H3 manifest ...')
_size_bad = []
for pat in _PATTERNS:
    dst = COMFY_DIR / 'models' / pat
    exp = _EXPECTED.get(pat, -1)
    if not dst.exists():
        _size_bad.append((pat, 0, exp))
        print(f'    MISSING  {dst.name}  (expected {exp/1024**3:.2f} GB)')
        continue
    got = dst.stat().st_size
    if got != exp:
        _size_bad.append((pat, got, exp))
        print(f'    WRONG    {dst.name}  (got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB)')
    else:
        print(f'    OK       {dst.name}  ({exp/1024**3:.2f} GB)')
if _size_bad:
    proc.terminate()
    msg = ('ComfyUI-side weights do not match the Comfy-Org manifest. '
           'This usually means a Drive FUSE copy was interrupted before completion.\n\n')
    for pat, got, exp in _size_bad:
        msg += f'  {pat}: got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB\n'
    msg += ('\nFix: delete the bad models and re-run STEP 2:\n'
            '  !rm -rf /content/drive/MyDrive/AI_Video/AEI_ComfyUI/models\n'
            '  # then re-run STEP 2 (it skips files that match the manifest size).\n')
    raise SystemExit(msg)

with urllib.request.urlopen(COMFY_URL + '/system_stats') as r:
    sysinfo = json.loads(r.read())
print(f'  ComfyUI ready: {sysinfo.get("system", {}).get("comfyui_version", "?")} on '
      f'{sysinfo.get("devices", [{}])[0].get("name", "?")}')

import builtins
builtins.AEI_COMFY_PROC = proc
builtins.AEI_COMFY_LOG = log_path
builtins.AEI_COMFY_URL = COMFY_URL
builtins.AEI_COMFY_DIR = COMFY_DIR

import re as _re
def _slug(s, maxlen=40):
    """Compress a prompt to a filesystem-safe slug, capped at maxlen chars.

    Strips non-alphanumerics, collapses runs of separators, lowercases. The
    result is short enough to fit a ComfyUI SaveVideo prefix and stays
    human-readable in the output filename.
    """
    s = _re.sub(r'[^a-zA-Z0-9_\-]+', '-', s).strip('-')
    s = _re.sub(r'-+', '-', s)
    return s[:maxlen].strip('-') or 'untitled'

builtins.H3_SLUG = _slug
print('  Stored AEI_COMFY_PROC / AEI_COMFY_URL in builtins for STEP 5/6 access.')


### STEP 5｜初始化生成引擎（**必要**，不會產生影片）
每次重新連線新的 Colab 執行階段，先完成 STEP 1、2、3，再執行這格。它會註冊 STEP 6/7 需要的 H3 工作流程，不啟動 Gradio。


In [ ]:
#@title STEP 5｜初始化 H3 工作流程引擎（STEP 6/7 必須執行）
# 這格取代原本 STEP 4 中「必要的工作流程產生器」，完全不啟動 Gradio。
# 每次重新建立 Colab 執行階段後，請先執行 STEP 1 → 2 → 3 → 5。
import os, json, time, shutil, builtins, urllib.request
from pathlib import Path
COMFY_URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR', '/content/drive/MyDrive/AI_Video/AEI_ComfyUI'))
try:
    with urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=5) as resp:
        _h3_stats = json.loads(resp.read())
except Exception as exc:
    raise RuntimeError('ComfyUI 尚未啟動，請先執行 STEP 3。') from exc

TURBO_LORA = 'minimax_h3_turbo_4step_ema_ckpt500_pruned_comfyui.safetensors'
LIGHTX2V_LORA = 'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'
LIGHTX2V_LORA_768P = 'minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors'

def _build_workflow(mode, prompt, width, height, length, steps, seed,
                    unet_name, clip_name, video_vae, audio_vae, first_frame=None,
                    last_frame=None, filename_prefix=None, turbo=False, sage=False,
                    spectrum=False, lightx2v=False, lora_name=None, lora_strength=1.0,
                    sigma_video=12.0, sigma_audio=6.0, sampler_name='res_multistep',
                    use_larryvrh_turbo=False, larryvrh_low_vram=False,
                    use_sol=False, use_t4_lowmem=False, scheduler_name='simple'):
    p = {}
    p["6"]  = {"class_type": "UNETLoader",   "inputs": {"unet_name": unet_name, "weight_dtype": "default"}}
    p["13"] = {"class_type": "CLIPLoader",   "inputs": {"clip_name": clip_name, "type": "minimax", "device": "default"}}
    p["11"] = {"class_type": "VAELoader",    "inputs": {"vae_name": video_vae}}
    p["24"] = {"class_type": "VAELoader",    "inputs": {"vae_name": audio_vae}}
    if mode == 'fl2va' and first_frame is not None:
        p["30"] = {"class_type": "LoadImage", "inputs": {"image": first_frame}}
    if mode == 'fl2va' and last_frame is not None:
        p["31"] = {"class_type": "LoadImage", "inputs": {"image": last_frame}}
    inputs = {"clip": ["13", 0], "vae": ["11", 0],
              "width": width, "height": height, "length": length, "prompt": prompt}
    if mode == 'fl2va' and first_frame is not None:
        inputs["first_frame"] = ["30", 0]
    if mode == 'fl2va' and last_frame is not None:
        inputs["last_frame"] = ["31", 0]
    p["104"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": inputs}

    # MODEL chain.
    #   UNETLoader
    #   -> MiniMaxChunkFeedForward (if use_t4_lowmem; halves peak MLP memory)
    #   -> LoraLoaderModelOnly (if turbo OR lightx2v AND NOT use_larryvrh_turbo)
    #   -> MiniMaxH3SigmaShift (if turbo OR lightx2v AND NOT use_larryvrh_turbo)
    #   -> MiniMaxH3TurboLoRA (if turbo AND use_larryvrh_turbo; replaces the
    #      LoraLoader + SigmaShift pair with the Larryvrh node, which auto-
    #      detects the pruned base and re-injects the adaln update at run
    #      time. The Turbo Sampler handles SigmaShift internally, so we
    #      don't need our own SigmaShift node in this path.)
    #   -> MiniMaxLowVRAMAttention (if use_t4_lowmem; mutually exclusive with
    #      sage/sol. Splits attention into head groups; output bit-exact.
    #      Works on every arch including sm_75 where Sage/Sol refuse to
    #      compile.)
    #   -> MiniMaxH3MemoryEfficientSageAttentionPatch (if sage)
    #   -> SpectrumApplyMiniMaxH3 (if spectrum)
    _model_out = ["6", 0]
    # Standard mode also needs a valid sampler node. LoRA/Turbo paths override it below.
    p["17"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": sampler_name}}
    if use_t4_lowmem:
        # kjnodes' MiniMaxChunkFeedForward: chunks the SwiGLU FFN over
        # the packed-token dim so kernel-internal activations scale by
        # the chunk count. Each chunk still uses ComfyUI's fused INT8
        # kernel when the base is quantized; activations are quantized
        # per-token so the output is bit-exact.
        #
        # chunks=16 is the value validated on T4 (16 GB VRAM) for the
        # 832x480 5s workflow with spectrum+turbo_lora. The comfy_kitchen
        # int8_linear kernel internally splits the int8 matmul OUTPUT
        # into ~10 parts (each ~3.5 GB at chunks=8, ~1.75 GB at
        # chunks=16). The cat at the end tries to allocate a buffer
        # matching the chunked output, which is what fires the OOM at
        # torch.cat(scaled_parts, dim=0). chunks=16 halves the per-part
        # allocation to ~1.75 GB and gives the kernel room to complete
        # the cat at <14.56 GiB ceiling. chunks=8 still OOMs on the
        # 832x480 5s workflow because the cat target is too large for
        # the active set; chunks=4 OOMs identically.
        #
        # Wallclock cost: ~30% vs chunks=2 (one kernel launch per
        # chunk). For roomy GPUs (L4+, A100) chunks=2 is faster but the
        # model fits anyway, so this override only fires when
        # use_t4_lowmem is set. If even chunks=16 OOMs, the user
        # should drop canvas to 704x400 (33% fewer tokens) rather than
        # bumping chunks higher — at chunks=32 the per-chunk launch
        # overhead dominates and wallclock exceeds 2 hours per clip.
        p["71"] = {"class_type": "MiniMaxChunkFeedForward",
                   "inputs": {"model": _model_out, "chunks": 16, "seq_threshold": 4096}}
        _model_out = ["71", 0]
    if use_larryvrh_turbo and turbo:
        # Larryvrh Turbo LoRA node: chains a single MODEL->MODEL node that
        # applies the LoRA (bypass by default; merge=low_vram mode folds
        # it into the weights, freeing ~1-2 GB of peak VRAM at the cost of
        # some sharpness on quantized bases). The strength field is
        # passed straight through.
        p["40"] = {"class_type": "MiniMaxH3TurboLoRA",
                   "inputs": {"model": _model_out,
                              "lora_name": lora_name or TURBO_LORA,
                              "strength": lora_strength,
                              "low_vram": larryvrh_low_vram}}
        _model_out = ["40", 0]
        if use_t4_lowmem:
            # kjnodes' MiniMaxLowVRAMAttention: splits the attention
            # call into head groups so the kernel's internal transients
            # (int8 q/k copies, fp32 accumulator) shrink by the chunk
            # count while the fused qkv buffer is alive, and frees the
            # qkv buffer / normed block input as soon as they're
            # consumed. Output is bit-exact with the unpatched model.
            # head_chunks=4 is kjnodes' default; this composes with the
            # built-in optimized_attention ComfyUI already picked (so
            # it works on sm_75 / T4). Mutually exclusive with
            # sage/sol — only one of these three is active per
            # workflow.
            p["70"] = {"class_type": "MiniMaxLowVRAMAttention",
                       "inputs": {"model": _model_out, "head_chunks": 4}}
            _model_out = ["70", 0]
        if sage:
            p["42"] = {"class_type": "MiniMaxH3MemoryEfficientSageAttentionPatch",
                       "inputs": {"model": _model_out, "sage_attention": 2}}
            _model_out = ["42", 0]
        if use_sol:
            # Saganaki22/ComfyUI-sol-attn's H3 Sol patch. The repo
            # ships two variants; the Memory-Efficient one is the
            # default (fixed tau, simpler). The eval's base20-sol
            # column uses the same node. We follow the recommended
            # chain order: Sage first, then Sol (Sol inherits Sage's
            # dense fallback for tokens/steps it can't accelerate).
            p["43"] = {"class_type": "MiniMaxH3MemoryEfficientSolAttentionPatch",
                       "inputs": {"model": _model_out, "enabled": True,
                                  "tau": 1.3, "min_tokens": 4096, "strict": False,
                                  "thresh_type": "diag", "int8_qk": False, "int8_pv": False,
                                  "sink_conditioning": "exact_kv", "dense_blocks": ""}}
            _model_out = ["43", 0]
        # The Turbo Sampler replaces KSamplerSelect. We add it as a
        # SAMPLER node and feed it to SamplerCustomAdvanced.
        p["17"] = {"class_type": "MiniMaxH3TurboSampler", "inputs": {}}
    elif turbo or lightx2v:
        p["40"] = {"class_type": "LoraLoaderModelOnly",
                   "inputs": {"model": _model_out,
                              "lora_name": lora_name or TURBO_LORA,
                              # ComfyUI v0.30 renamed the LoRA-strength input on
                              # LoraLoaderModelOnly from `strength` to
                              # `strength_model` (to disambiguate from
                              # LoraLoader's strength_clip). The old name
                              # triggers a `Required input is missing`
                              # validation error on v0.30+ (caught on T4
                              # run with the drbaph path).
                              "strength_model": lora_strength}}
        _model_out = ["40", 0]
        # MiniMaxH3SigmaShift. The drbaph turbo_lora uses 12/6 (house);
        # the lightx2v _ersde variants use the upstream default 12/3
        # (matches Kijai's recommended recipe and the eval's lx4-kijai
        # / lx4-ersde10 columns). lightx2v without _ersde uses 12/6
        # (matches the eval's lx4-house / lx8-house columns). The
        # caller passes sigma_video and sigma_audio explicitly.
        p["41"] = {"class_type": "MiniMaxH3SigmaShift",
                   "inputs": {"model": _model_out,
                              "shift_video": sigma_video,
                              "shift_audio": sigma_audio}}
        _model_out = ["41", 0]
        if use_t4_lowmem:
            # kjnodes' MiniMaxLowVRAMAttention — see larryvrh branch
            # above for the rationale. Identical node id (70) and
            # inputs in both branches; we always reset _model_out so
            # the spectrum / BasicGuider nodes consume the same chain
            # regardless of which turbo backend is active.
            p["70"] = {"class_type": "MiniMaxLowVRAMAttention",
                       "inputs": {"model": _model_out, "head_chunks": 4}}
            _model_out = ["70", 0]
        if sage:
            # KJNodes' SageAttention patch. Running after SigmaShift keeps the
            # LoRA-modified model's sigmas intact while still patching the
            # attention ops. sage_attention=2 selects the kjnodes default
            # which is auto-detected at install time (sparse if available).
            p["42"] = {"class_type": "MiniMaxH3MemoryEfficientSageAttentionPatch",
                       "inputs": {"model": _model_out, "sage_attention": 2}}
            _model_out = ["42", 0]
        if use_sol:
            # Sol runs after Sage (Sol inherits Sage's dense fallback).
            p["43"] = {"class_type": "MiniMaxH3MemoryEfficientSolAttentionPatch",
                       "inputs": {"model": _model_out, "enabled": True,
                                  "tau": 1.3, "min_tokens": 4096, "strict": False,
                                  "thresh_type": "diag", "int8_qk": False, "int8_pv": False,
                                  "sink_conditioning": "exact_kv", "dense_blocks": ""}}
            _model_out = ["43", 0]
        p["17"] = {"class_type": "KSamplerSelect","inputs": {"sampler_name": sampler_name}}
    if spectrum:
        # xmarre's Spectral Apply MiniMax H3. The preliminary defaults
        # from the upstream README are blend_weight=0.50, degree=1,
        # ridge_lambda=0.10, window_size=2.0, flex_window=0.75,
        # warmup_steps=1, tail_actual_steps=1, max_history=8, and
        # bootstrap_first_forecast=True. The eval uses these for the
        # spec columns. sampler_name must be on the Spectrum allowlist
        # (euler, res_multistep, res_multistep_cfg_pp); if the user
        # picked er_sde we set `enabled=False` so the node becomes a
        # pass-through and Spectrum's natural fallback to native
        # doesn't fight the er_sde schedule. The trade-off: er_sde
        # + spectrum still gets the same wall-clock as er_sde alone
        # (Spectrum is no-op when disabled) but the user can switch
        # to res_multistep to re-enable Spectrum without changing
        # anything else.
        _spectrum_enabled = sampler_name in ('euler', 'res_multistep', 'res_multistep_cfg_pp')
        p["50"] = {"class_type": "SpectrumApplyMiniMaxH3",
                   "inputs": {"model": _model_out,
                              "enabled": _spectrum_enabled,
                              "blend_weight": 0.50,
                              "degree": 1,
                              "ridge_lambda": 0.10,
                              "window_size": 2.0,
                              "flex_window": 0.75,
                              "warmup_steps": 1,
                              "tail_actual_steps": 1,
                              "max_history": 8,
                              "debug": False,
                              "history_storage": "system_ram",
                              "bootstrap_first_forecast": _spectrum_enabled}}
        _model_out = ["50", 0]

    p["16"] = {"class_type": "BasicGuider",   "inputs": {"model": _model_out, "conditioning": ["104", 0]}}
    # p["17"] is set above (KSamplerSelect for drbaph/lightx2v paths,
    # MiniMaxH3TurboSampler for the larryvrh turbo path).
    p["9"]  = {"class_type": "BasicScheduler","inputs": {"model": _model_out, "scheduler": scheduler_name,
                                                          "steps": steps, "denoise": 1}}
    p["15"] = {"class_type": "RandomNoise",   "inputs": {"noise_seed": seed}}
    p["14"] = {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["15", 0], "guider": ["16", 0], "sampler": ["17", 0],
                          "sigmas": ["9", 0], "latent_image": ["104", 1]}}
    p["10"] = {"class_type": "VAEDecode",      "inputs": {"samples": ["14", 0], "vae": ["11", 0]}}
    p["23"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["14", 0], "vae": ["24", 0]}}
    p["91"] = {"class_type": "CreateVideo",    "inputs": {"images": ["10", 0], "audio": ["23", 0], "fps": 24}}
    p["92"] = {"class_type": "SaveVideo",      "inputs": {"video": ["91", 0],
                                                            "filename_prefix": filename_prefix or f"video/H3_{mode}",
                                                           "format": "auto", "codec": "auto"}}
    return {"prompt": p}


def _build_workflow_r2v(prompt, width, height, length, steps, seed,
                        unet_name, clip_name, video_vae, audio_vae,
                        ref_image_paths=None,    # list of up to 9 paths
                        ref_video_paths=None,    # list of up to 3 paths (mp4)
                        ref_video_audios=None,   # list of up to 3 paths (wav/mp3)
                        ref_audios=None,         # list of up to 3 paths (wav/mp3)
                        ref_image_size='match',  # 'match' or 'max'
                        filename_prefix=None, scheduler_name='simple',
                        sampler_name='res_multistep'):
    """Build the official ComfyUI Ref2Vid (R2V) workflow for MiniMax-H3.

    Mirrors the official template at
    https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_r2v.json

    The conditioning node is MiniMaxH3ReferenceToVideo (different from
    the MiniMaxH3ImageToVideo used by fl2v). It accepts up to 9 images,
    3 videos, 3 video-paired audios, and 3 standalone audios. Slots
    not connected are simply ignored.

    The prompt uses reference tags in connection order: <Picture 1>,
    <Video 1>, <Audio 1>, etc. (1-indexed, in the exact order the
    references are connected). The model is sensitive to exact tag
    wording.
    """
    p = {}
    p['6']  = {'class_type': 'UNETLoader',   'inputs': {'unet_name': unet_name, 'weight_dtype': 'default'}}
    p['13'] = {'class_type': 'CLIPLoader',   'inputs': {'clip_name': clip_name, 'type': 'minimax', 'device': 'default'}}
    p['11'] = {'class_type': 'VAELoader',    'inputs': {'vae_name': video_vae}}
    p['24'] = {'class_type': 'VAELoader',    'inputs': {'vae_name': audio_vae}}

    # Reference image uploads (up to 9). Each gets its own LoadImage node.
    if ref_image_paths:
        for i, img_path in enumerate(ref_image_paths[:9]):
            fname = _stage_comfy_input(img_path) if '_stage_comfy_input' in globals() else Path(img_path).name
            node_id = str(30 + i)
            p[node_id] = {'class_type': 'LoadImage',
                          'inputs': {'image': fname, 'upload': 'image'}}

    # Reference video uploads (up to 3). VHS_LoadVideo produces IMAGE-batch
    # output from an mp4 file.
    if ref_video_paths:
        for i, vid_path in enumerate(ref_video_paths[:3]):
            fname = _stage_comfy_input(vid_path) if '_stage_comfy_input' in globals() else Path(vid_path).name
            node_id = str(50 + i)
            p[node_id] = {'class_type': 'VHS_LoadVideo',
                          'inputs': {'video': fname, 'force_rate': 0, 'force_size': 'Disabled'}}

    # Reference audios (up to 3 + up to 3 paired with videos).
    if ref_audios:
        for i, aud_path in enumerate(ref_audios[:3]):
            fname = _stage_comfy_input(aud_path) if '_stage_comfy_input' in globals() else Path(aud_path).name
            node_id = str(60 + i)
            p[node_id] = {'class_type': 'LoadAudio',
                          'inputs': {'audio': fname}}
    if ref_video_audios:
        for i, aud_path in enumerate(ref_video_audios[:3]):
            fname = _stage_comfy_input(aud_path) if '_stage_comfy_input' in globals() else Path(aud_path).name
            node_id = str(70 + i)
            p[node_id] = {'class_type': 'LoadAudio',
                          'inputs': {'audio': fname}}

    # The conditioning node. R2V wants all the references as named
    # slots (ref_image_0, ref_image_1, ...). The official template
    # wires only the first 2 ref_images; we wire all provided refs.
    inputs = {
        'clip': ['13', 0],
        'vae': ['11', 0],
        'audio_vae': ['24', 0],
        'prompt': prompt,
        'width': width,
        'height': height,
        'length': length,
        'ref_image_size': ref_image_size or 'match',
    }
    if ref_image_paths:
        for i in range(min(len(ref_image_paths), 9)):
            inputs[f'ref_images.ref_image_{i}'] = [str(30 + i), 0]
    if ref_video_paths:
        for i in range(min(len(ref_video_paths), 3)):
            inputs[f'ref_videos.ref_video_{i}'] = [str(50 + i), 0]
    if ref_video_audios:
        for i in range(min(len(ref_video_audios), 3)):
            inputs[f'ref_video_audios.ref_video_audio_{i}'] = [str(70 + i), 0]
    if ref_audios:
        for i in range(min(len(ref_audios), 3)):
            inputs[f'ref_audios.ref_audio_{i}'] = [str(60 + i), 0]

    p['104'] = {'class_type': 'MiniMaxH3ReferenceToVideo',
                'inputs': inputs}

    # The R2V template is simpler than the fl2v turbo path - no LoRA,
    # no SigmaShift, no Spectrum, no Sage/Sol patches, no larryvrh.
    # Just the base UNet through the standard sampler.
    p['16'] = {'class_type': 'BasicGuider',
               'inputs': {'model': ['6', 0], 'conditioning': ['104', 0]}}
    p['17'] = {'class_type': 'KSamplerSelect',
               'inputs': {'sampler_name': sampler_name}}
    p['9']  = {'class_type': 'BasicScheduler',
               'inputs': {'model': ['6', 0], 'scheduler': scheduler_name, 'steps': steps, 'denoise': 1}}
    p['15'] = {'class_type': 'RandomNoise',
               'inputs': {'noise_seed': seed}}
    p['14'] = {'class_type': 'SamplerCustomAdvanced',
               'inputs': {'noise': ['15', 0], 'guider': ['16', 0], 'sampler': ['17', 0],
                          'sigmas': ['9', 0], 'latent_image': ['104', 1]}}
    p['10'] = {'class_type': 'VAEDecode',      'inputs': {'samples': ['14', 0], 'vae': ['11', 0]}}
    p['23'] = {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['14', 0], 'vae': ['24', 0]}}
    p['91'] = {'class_type': 'CreateVideo',    'inputs': {'images': ['10', 0], 'audio': ['23', 0], 'fps': 24}}
    p['92'] = {'class_type': 'SaveVideo',      'inputs': {'video': ['91', 0],
                                                            'filename_prefix': filename_prefix or 'video/H3_r2v',
                                                            'format': 'auto', 'codec': 'auto'}}
    return {'prompt': p}


def _stage_comfy_input(local_path):
    """Ensure a file is present in ComfyUI's input directory and return its filename."""
    if not local_path:
        return None
    # Handle local filepath or an uploaded file-like object
    path_str = getattr(local_path, 'name', None) or (local_path if isinstance(local_path, str) else str(local_path))
    p = Path(path_str)
    if not p.exists():
        return None
    input_dir = COMFY_DIR / 'input'
    input_dir.mkdir(parents=True, exist_ok=True)
    dest = input_dir / p.name
    try:
        if p.resolve() != dest.resolve():
            shutil.copy2(p, dest)
    except Exception as e:
        print(f'  [warn] failed to copy {p} to ComfyUI input: {e}')
    return p.name


# 發布給 STEP 6 / STEP 7 使用；STEP 4 已移除。
builtins._build_workflow = _build_workflow
builtins._build_workflow_r2v = _build_workflow_r2v
builtins._TURBO_LORA = TURBO_LORA
builtins._LIGHTX2V_LORA = LIGHTX2V_LORA
builtins._LIGHTX2V_LORA_768P = LIGHTX2V_LORA_768P
print('✅ H3 工作流程已初始化：t2va / fl2va / r2v。無須 Gradio。')
for _h3_device in _h3_stats.get('devices', []):
    _vram = _h3_device.get('vram_total', 0) / (1024**3)
    print(f'  GPU：{_h3_device.get("name", "unknown")}，總顯存 {_vram:.1f} GB')
print('下一步：需要上傳圖片時執行 STEP 5.5；生成影片請執行 STEP 6。')


# ===== H3_SHARED_HELPERS_BEGIN =====
# STEP 6 / STEP 7 共用的工具。放在 STEP 5，因為兩格都必須先跑 STEP 5。
import hashlib as _h3_hashlib
import mimetypes as _h3_mimetypes
import re as _h3_re
import urllib.error as _h3_urlerror
import urllib.parse as _h3_urlparse
import uuid as _h3_uuid
import requests as _h3_requests

H3_REF_DIR = COMFY_DIR.parent / 'H3_Reference_Images'
H3_FINAL_DIR = COMFY_DIR.parent / 'Outputs'
_H3_VIDEO_EXT = ('.mp4', '.webm', '.mov')


def _h3_fmt(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'


def _h3_slug(text, maxlen=40):
    text = _h3_re.sub(r'[^a-zA-Z0-9_-]+', '-', str(text)).strip('-')
    return _h3_re.sub(r'-+', '-', text)[:maxlen].strip('-') or 'untitled'


def _h3_canvas(label):
    """'960 x 544 · 16:9 standard' -> (960, 544)。也接受自己輸入的 '1280 x 720'。"""
    m = _h3_re.match(r'\s*(\d+)\s*[xX×]\s*(\d+)', str(label or ''))
    if not m:
        raise ValueError(f'看不懂畫布設定：{label!r}（格式要像 960 x 544）')
    return int(m.group(1)), int(m.group(2))


def _h3_length(duration):
    """秒數 -> H3 合法的影格數（24 fps，且除以 17 餘 5）。"""
    length = int(duration) * 24
    while length % 17 != 5:
        length += 1
    return length


def _h3_auto_steps(gen_mode='fl2va'):
    """依 STEP 1 選的 MODE 回傳建議步數（與 STEP 1 的對照表一致）。"""
    if gen_mode == 'r2v':
        return 20  # r2v 固定用基礎模型，不吃加速 LoRA
    seg = getattr(builtins, 'H3_MODE', 'standard').split('+')
    turbo = seg[-1]
    spectrum = 'spectrum' in seg
    if turbo in ('turbo_lora', 'turbo_lora_sage'):
        if getattr(builtins, 'H3_TURBO_BACKEND', 'drbaph') == 'larryvrh':
            return 8 if spectrum else 6
        if spectrum:
            return 12 if turbo == 'turbo_lora' else 10
        return 10 if turbo == 'turbo_lora' else 8
    if turbo.startswith('lightx2v_'):
        base = {'lightx2v_4step': 4, 'lightx2v_8step': 8,
                'lightx2v_4step_ersde': 5, 'lightx2v_8step_ersde': 9}.get(turbo, 4)
        return base + (2 if spectrum else 0)
    if turbo in ('spectrum', 'fp8_dense'):
        return 16
    return 20


def _h3_resolve(raw, what='檔案'):
    """把表單／JSON 裡填的路徑變成實際存在的檔案。
    可以只填檔名（到 H3_Reference_Images 找），也可以填完整路徑。空白回傳 None。"""
    raw = str(raw or '').strip().strip('"\'').strip()
    if not raw:
        return None
    p = Path(raw)
    if p.is_absolute():
        candidates = [p, H3_REF_DIR / p.name]
    else:
        candidates = [H3_REF_DIR / raw, COMFY_DIR / 'input' / raw, Path('/content') / raw]
    for c in candidates:
        if c.is_file():
            return c
    have = []
    if H3_REF_DIR.is_dir():
        have = sorted(x.name for x in H3_REF_DIR.iterdir() if x.is_file())[:40]
    raise ValueError(f'找不到{what}：{raw}\n    {H3_REF_DIR} 裡目前有：'
                     + ('、'.join(have) if have else '（空的）'))


def _h3_resolve_list(raw, what='檔案'):
    """逗號分隔的字串或 JSON 清單 -> 檔案清單。"""
    if isinstance(raw, (list, tuple)):
        items = [str(x) for x in raw]
    else:
        items = str(raw or '').split(',')
    out = []
    for item in items:
        p = _h3_resolve(item, what)
        if p is not None:
            out.append(p)
    return out


def _h3_upload(path):
    """把首幀／尾幀送進 ComfyUI，回傳 ComfyUI 端的檔名。"""
    path = Path(path)
    mime = _h3_mimetypes.guess_type(path.name)[0] or 'image/png'
    with path.open('rb') as f:
        r = _h3_requests.post(COMFY_URL + '/upload/image',
                              files={'image': (path.name, f, mime)},
                              data={'type': 'input', 'overwrite': 'true'}, timeout=120)
    r.raise_for_status()
    return r.json()['name']


def _h3_wait(prompt_id, poll=3, on_progress=None):
    """等 ComfyUI 跑完。偶爾查不到進度不會中斷，只有 ComfyUI 真的掛掉才報錯。"""
    out_dir = COMFY_DIR / 'output'
    log_path = Path(getattr(builtins, 'AEI_COMFY_LOG', COMFY_DIR / 'comfyui.log'))
    log_pos = log_path.stat().st_size if log_path.exists() else 0
    t0 = time.time()
    last_report = 0
    fails = 0
    while True:
        entry = None
        try:
            hist = _h3_requests.get(f'{COMFY_URL}/history/{prompt_id}', timeout=30).json()
            entry = hist.get(prompt_id)
            fails = 0
        except Exception as exc:
            fails += 1
            proc = getattr(builtins, 'AEI_COMFY_PROC', None)
            if proc is not None and proc.poll() is not None:
                raise RuntimeError('ComfyUI 已經停止（常見原因是記憶體不足）。'
                                   '請用 STEP 8 看日誌，再重跑 STEP 3 和 STEP 5。') from exc
            if fails == 1 or fails % 40 == 0:
                print(f'   （暫時查不到進度，第 {fails} 次，繼續等待…）', flush=True)
            if fails >= 400:
                raise RuntimeError('ComfyUI 已經連續很久沒有回應，請用 STEP 8 看日誌。') from exc
        if entry:
            status = entry.get('status', {})
            if status.get('completed'):
                break
            if status.get('error') or status.get('status_str') == 'error':
                raise RuntimeError('ComfyUI 生成失敗：'
                                   + json.dumps(status.get('messages', []), ensure_ascii=False)[:2000])
        new_lines = []
        try:
            if log_path.exists():
                size = log_path.stat().st_size
                if size > log_pos:
                    with log_path.open('rb') as f:
                        f.seek(log_pos)
                        new_lines = f.read().decode('utf-8', errors='replace').splitlines()
                    log_pos = size
        except OSError:
            pass
        if on_progress is not None:
            try:
                on_progress(time.time() - t0, new_lines)
            except Exception:
                pass
        tail = ' | '.join(new_lines[-3:])[:200] if new_lines else ''
        if tail or time.time() - last_report > 300:
            last_report = time.time()
            print(f'   ... {_h3_fmt(time.time() - t0)}' + (f'  {tail}' if tail else ''), flush=True)
        time.sleep(poll)
    elapsed = time.time() - t0

    paths = []
    for node_out in entry.get('outputs', {}).values():
        for kind in ('videos', 'images', 'audio'):
            for out in node_out.get(kind) or []:
                if not isinstance(out, dict) or not out.get('filename'):
                    continue
                fn = out['filename']
                sub = out.get('subfolder') or ''
                candidates = [out_dir / sub / fn, out_dir / fn, out_dir / 'video' / fn]
                local = None
                for attempt in range(7):
                    local = next((c for c in candidates if c.exists()), None)
                    if local is None and attempt == 3:
                        local = next(iter(out_dir.glob(f'*/{fn}')), None)
                    if local is not None:
                        break
                    time.sleep(2)
                if local is None:
                    # 雲端硬碟還沒同步出檔案時，直接跟 ComfyUI 要一份。
                    local = candidates[0]
                    local.parent.mkdir(parents=True, exist_ok=True)
                    url = (f'{COMFY_URL}/view?filename={_h3_urlparse.quote(fn)}'
                           f'&subfolder={_h3_urlparse.quote(sub)}&type={out.get("type", "output")}')
                    for attempt in range(3):
                        try:
                            with urllib.request.urlopen(url, timeout=120) as resp, open(local, 'wb') as f:
                                shutil.copyfileobj(resp, f)
                            break
                        except _h3_urlerror.HTTPError as e:
                            if e.code == 404 and attempt < 2:
                                time.sleep(3)
                                continue
                            raise
                paths.append(local)
    if not paths:
        raise RuntimeError('ComfyUI 回報完成，但沒有找到輸出檔案；請用 STEP 8 看日誌。')
    return paths, elapsed


def _h3_generate(gen_mode, prompt, width, height, length, steps, seed, tag='clip',
                 first=None, last=None, ref_images=(), ref_videos=(), ref_audios=(),
                 ref_image_size='match', poll=3, on_progress=None):
    """組工作流程、送進 ComfyUI、等它完成。回傳 (影片路徑, 耗時秒數)。"""
    mode = getattr(builtins, 'H3_MODE', 'standard')
    seg = mode.split('+')
    turbo_side = seg[-1]
    lightx2v = turbo_side.startswith('lightx2v_')
    is_turbo = turbo_side in ('turbo_lora', 'turbo_lora_sage')
    prefix = f'video/H3_{tag}_{width}x{height}_f{length}_s{steps}_seed{seed}_{int(time.time())}'
    clip = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
    vvae = 'minimax_h3_video_vae_fp16.safetensors'
    avae = 'minimax_h3_audio_vae_fp32.safetensors'

    if gen_mode == 'r2v':
        # r2v 固定用 ref2va 基礎模型，不套用 STEP 1 的加速模式。
        wf = _build_workflow_r2v(
            prompt=prompt, width=width, height=height, length=length, steps=steps, seed=seed,
            unet_name='minimax_h3_ref2va_pruned_int8_convrot.safetensors',
            clip_name=clip, video_vae=vvae, audio_vae=avae,
            ref_image_paths=[str(p) for p in ref_images],
            ref_video_paths=[str(p) for p in ref_videos],
            ref_audios=[str(p) for p in ref_audios],
            ref_video_audios=[],
            ref_image_size=ref_image_size or 'match',
            filename_prefix=prefix,
            sampler_name='res_multistep', scheduler_name='simple')
    else:
        unet = ('minimax_h3_fl2va_pruned_fp8_scaled.safetensors' if mode.startswith('fp8_')
                else 'minimax_h3_fl2va_pruned_int8_convrot.safetensors')
        if turbo_side == 'lightx2v_4step_768p':
            lora, strength, sigma_v, sigma_a = builtins._LIGHTX2V_LORA_768P, 1.0, 6.0, 3.0
        elif lightx2v:
            ersde = turbo_side.endswith('_ersde')
            lora = builtins._LIGHTX2V_LORA
            strength = 0.75 if ersde else 1.0
            sigma_v, sigma_a = (12.0, 3.0) if ersde else (12.0, 6.0)
        else:
            lora, strength, sigma_v, sigma_a = builtins._TURBO_LORA, 1.0, 12.0, 6.0
        first_name = _h3_upload(first) if (gen_mode == 'fl2va' and first) else None
        last_name = _h3_upload(last) if (gen_mode == 'fl2va' and last) else None
        wf = _build_workflow(
            gen_mode, prompt, width, height, length, steps, seed,
            unet, clip, vvae, avae,
            first_frame=first_name, last_frame=last_name,
            filename_prefix=prefix,
            turbo=is_turbo,
            sage=turbo_side == 'turbo_lora_sage',
            spectrum='spectrum' in seg,
            lightx2v=lightx2v,
            lora_name=lora, lora_strength=strength,
            sigma_video=sigma_v, sigma_audio=sigma_a,
            sampler_name=getattr(builtins, 'H3_SAMPLER', 'res_multistep'),
            use_larryvrh_turbo=(is_turbo and getattr(builtins, 'H3_TURBO_BACKEND', 'drbaph') == 'larryvrh'),
            larryvrh_low_vram=getattr(builtins, 'H3_LOW_VRAM', False),
            use_sol=getattr(builtins, 'H3_USE_SOL_ATTENTION', False),
            use_t4_lowmem=getattr(builtins, 'H3_USE_T4_LOWMEM', False),
            scheduler_name='simple')

    r = _h3_requests.post(COMFY_URL + '/prompt',
                          json={'prompt': wf['prompt'], 'client_id': str(_h3_uuid.uuid4())}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f'ComfyUI 不接受這個工作流程：{r.status_code} {r.text[:1000]}')
    prompt_id = r.json()['prompt_id']
    print(f'   已送出（{prompt_id[:8]}），生成中…', flush=True)
    paths, elapsed = _h3_wait(prompt_id, poll=poll, on_progress=on_progress)
    video = next((p for p in paths if p.suffix.lower() in _H3_VIDEO_EXT), paths[0])
    return video, elapsed


def _h3_publish(src, stem, seed, info, dest_dir=None):
    """把成品複製到 AI_Video/Outputs（檔名好認），旁邊存一份同名 .txt 生成紀錄。"""
    src = Path(src)
    dest_dir = Path(dest_dir) if dest_dir else H3_FINAL_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    safe = _h3_re.sub(r'[\\/:*?"<>|\s]+', '_', str(stem)).strip('_.') or 'untitled'
    base = f'{safe}_seed{seed}'
    dest = dest_dir / f'{base}{src.suffix}'
    n = 2
    while dest.exists():
        dest = dest_dir / f'{base}_{n}{src.suffix}'
        n += 1
    shutil.copy2(src, dest)
    lines = [f'{k}：{v}' for k, v in info.items() if k != '提示詞' and v not in (None, '', [])]
    lines += [f'ComfyUI 原始檔：{src}', '', '提示詞：', str(info.get('提示詞', ''))]
    dest.with_suffix('.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return dest


def _h3_preview(path, max_mb=30):
    """在 notebook 裡直接播放。太大的檔案不內嵌，避免 notebook 變得很肥。"""
    from IPython.display import display, Video
    path = Path(path)
    size_mb = path.stat().st_size / 1024**2
    if path.suffix.lower() not in _H3_VIDEO_EXT:
        return
    if size_mb > max_mb:
        print(f'   （檔案 {size_mb:.0f} MB，不內嵌預覽，請到雲端硬碟看。）')
        return
    display(Video(str(path), embed=True, width=640, html_attributes='controls loop'))


def _h3_disconnect():
    """同步雲端硬碟後釋放執行階段，停止扣運算單元。"""
    print('\n準備中斷執行階段：先停止 ComfyUI，再把檔案同步到雲端硬碟…', flush=True)
    proc = getattr(builtins, 'AEI_COMFY_PROC', None)
    try:
        if proc is not None and proc.poll() is None:
            proc.terminate()
            proc.wait(timeout=30)
    except Exception:
        pass
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        print('雲端硬碟已同步。', flush=True)
    except Exception as exc:
        print(f'（同步雲端硬碟時出現訊息：{exc}）', flush=True)
    from google.colab import runtime
    runtime.unassign()


print('✅ STEP 6／7 共用工具已載入。成品會複製到：' + str(H3_FINAL_DIR))
# ===== H3_SHARED_HELPERS_END =====


### STEP 5.5｜從電腦**一次選多張圖片**上傳至 Google Drive（選用）
點 ▶ 後可在選檔視窗用 Ctrl／Shift 多選。每張檔案會顯示完整 Drive 路徑；同名檔案自動改名避免覆蓋。`fl2va` 複製一張到 `IMAGE_PATH`，`r2v` 多張路徑以英文逗號放在 `REF_IMAGES`。


In [ ]:
#@title STEP 5.5｜批次上傳圖片／參考素材到 Google Drive（可選多檔）
# files.upload() 本來就可一次選多檔；在選檔視窗用 Ctrl/Shift 多選。
# 上傳只儲存檔案，不會呼叫 H3 推論，也不會消耗影片生成次數。
from google.colab import files
from pathlib import Path
import shutil

import builtins
UPLOAD_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR', '/content/drive/MyDrive/AI_Video/AEI_ComfyUI')).parent / 'H3_Reference_Images'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print('請選擇一個或多個圖片／影片／音訊檔（可 Ctrl／Shift 多選）。')
uploaded = files.upload()
saved_paths = []
for filename in uploaded:
    src = Path(filename)
    if not src.is_file():
        print(f'⚠ 找不到剛上傳的檔案：{filename}')
        continue
    dest = UPLOAD_DIR / src.name
    # 不覆蓋上一次上傳的同名檔案。
    suffix = 2
    while dest.exists():
        dest = UPLOAD_DIR / f'{src.stem}_{suffix}{src.suffix}'
        suffix += 1
    shutil.copy2(src, dest)
    saved_paths.append(str(dest))
    print(f'✅ {len(saved_paths)}. {dest}')

if saved_paths:
    print(f'\n本次共儲存 {len(saved_paths)} 個檔案。')
    print('fl2va：複製單張圖片路徑到 STEP 6 的 IMAGE_PATH。')
    print('r2v：將要參考的多張圖片路徑，以英文逗號分隔，貼入 REF_IMAGES。')
    print('下面是本次路徑清單（依上傳順序）：')
    print('\n'.join(saved_paths))
    print('\nr2v 一次複製用：')
    print(', '.join(saved_paths))
else:
    print('未選取檔案。')


### STEP 6｜單支影片生成（主要使用方式）
必須先執行 STEP 5。在表單填好後按 ▶。

- `SHOT_NAME`：鏡頭代號（例如 `S01`），成品檔名會用它。
- 圖片欄位可以**只填檔名**，會自動到 `H3_Reference_Images` 找；貼完整路徑也可以。
- `USE_LAST_FRAME`：取消勾選就不使用尾幀，不用清空欄位。
- `STEPS = 0`：依 STEP 1 選的模式自動決定步數。`VARIATIONS`：一次生成多個不同 seed 的版本。
- 成品會複製到 `AI_Video/Outputs`，旁邊有同名 `.txt` 生成紀錄（完整提示詞、seed、參數），並直接在下方播放。
- `AUTO_DISCONNECT`：勾選後，生成完會同步雲端硬碟並中斷執行階段，停止扣運算單元。

In [ ]:
#@title STEP 6｜單支影片生成
#@markdown **鏡頭代號**：成品檔名會用它，例如 `S01`。留空就用提示詞開頭幾個字。
SHOT_NAME = '' #@param {type:"string"}
PROMPT = "A red fox walks through a snowy pine forest at dawn, slow dolly push-in, soft wind and snow crunching underfoot." #@param {type:"string"}
GEN_MODE = 't2va' #@param ['t2va', 'fl2va', 'r2v']
#@markdown ---
#@markdown **fl2va 用**：可以只填檔名（會到 `H3_Reference_Images` 找），也可以貼完整路徑。
IMAGE_PATH = '' #@param {type:"string"}
LAST_IMAGE_PATH = '' #@param {type:"string"}
#@markdown 取消勾選就不使用尾幀，不用清空上面的欄位。
USE_LAST_FRAME = True #@param {type:"boolean"}
#@markdown ---
#@markdown **r2v 用**：多個檔案用英文逗號分隔（圖片最多 9、影片 3、音訊 3）。
REF_IMAGES = '' #@param {type:'string'}
REF_VIDEOS = '' #@param {type:'string'}
REF_AUDIOS = '' #@param {type:'string'}
REF_IMAGE_SIZE = 'match' #@param ['match', 'max']
#@markdown ---
CANVAS = "832 x 480 · 16:9 fast (default)" #@param ['1152 x 640 · 16:9 wide', '960 x 544 · 16:9 standard', '832 x 480 · 16:9 fast (default)', '704 x 400 · 16:9 cheaper', '576 x 320 · 16:9 minimal', '640 x 1152 · 9:16 wide', '544 x 960 · 9:16 standard', '480 x 832 · 9:16 fast', '400 x 704 · 9:16 cheaper', '320 x 576 · 9:16 minimal', '768 x 768 · 1:1 full', '576 x 576 · 1:1 fast', '768 x 576 · 4:3', '576 x 768 · 3:4', '1344 x 544 · ~21:9', '1024 x 416 · ~21:9 cheaper'] {"allow-input": true}
DURATION = 5 #@param {type:"slider", min:2, max:14, step:1}
#@markdown **STEPS = 0**：依 STEP 1 選的模式自動決定步數。
STEPS = 0 #@param {type:"slider", min:0, max:40, step:1}
#@markdown **SEED = 0**：隨機。
SEED = 0 #@param {type:"integer"}
#@markdown 一次生成幾個版本（每個版本用不同 seed）。
VARIATIONS = 1 #@param {type:"slider", min:1, max:8, step:1}
#@markdown 生成完直接在下方播放。
SHOW_PREVIEW = True #@param {type:"boolean"}
#@markdown 勾選後，全部生成完會同步雲端硬碟並中斷執行階段（停止扣運算單元）。
AUTO_DISCONNECT = False #@param {type:"boolean"}

import builtins, random, time

if '_h3_generate' not in globals():
    raise SystemExit('請先執行 STEP 5（每次重開執行階段都要跑 STEP 1、2、3、5），再執行 STEP 6。')

# ── 檢查輸入，有問題在開始生成前就講清楚 ──
try:
    WIDTH, HEIGHT = _h3_canvas(CANVAS)
    _first = _last = None
    _refs_i, _refs_v, _refs_a = [], [], []
    if GEN_MODE == 'fl2va':
        _first = _h3_resolve(IMAGE_PATH, '首幀圖片')
        if _first is None:
            raise ValueError('fl2va 需要首幀：請在 IMAGE_PATH 填檔名或完整路徑。')
        if USE_LAST_FRAME:
            _last = _h3_resolve(LAST_IMAGE_PATH, '尾幀圖片')
    elif GEN_MODE == 'r2v':
        _refs_i = _h3_resolve_list(REF_IMAGES, '參考圖片')
        _refs_v = _h3_resolve_list(REF_VIDEOS, '參考影片')
        _refs_a = _h3_resolve_list(REF_AUDIOS, '參考音訊')
        if not (_refs_i or _refs_v or _refs_a):
            raise ValueError('r2v 至少要一個參考素材：請填 REF_IMAGES、REF_VIDEOS 或 REF_AUDIOS。')
        if len(_refs_i) > 9 or len(_refs_v) > 3 or len(_refs_a) > 3:
            raise ValueError('r2v 最多 9 張圖片、3 支影片、3 段音訊。')
except ValueError as _e:
    raise SystemExit(str(_e)) from None

LENGTH = _h3_length(DURATION)
_auto = _h3_auto_steps(GEN_MODE)
_steps = int(STEPS) if int(STEPS) > 0 else _auto
_name = SHOT_NAME.strip() or _h3_slug(PROMPT, 40)
_mode = getattr(builtins, 'H3_MODE', 'standard')

print('=' * 72)
print(f' 鏡頭     ：{_name}')
print(f' 生成方式 ：{GEN_MODE}（STEP 1 模式：{_mode}）')
print(f' 畫布     ：{WIDTH} x {HEIGHT}，{LENGTH} 幀 = {LENGTH / 24:.2f} 秒')
print(f' 步數     ：{_steps}' + ('（自動）' if int(STEPS) <= 0 else ''))
if int(STEPS) > 0 and int(STEPS) != _auto:
    print(f'            提醒：目前模式的建議步數是 {_auto}，想自動套用就把 STEPS 設成 0。')
if GEN_MODE == 'fl2va':
    print(f' 首幀     ：{_first}')
    if _last:
        print(f' 尾幀     ：{_last}')
    elif LAST_IMAGE_PATH.strip():
        print(' 尾幀     ：（有填，但 USE_LAST_FRAME 沒勾，這次不使用）')
    else:
        print(' 尾幀     ：（無）')
if GEN_MODE == 'r2v':
    for _n, _p in enumerate(_refs_i, 1):
        print(f' <Picture {_n}>：{_p.name}')
    for _n, _p in enumerate(_refs_v, 1):
        print(f' <Video {_n}>  ：{_p.name}')
    for _n, _p in enumerate(_refs_a, 1):
        print(f' <Audio {_n}>  ：{_p.name}')
print(f' 版本數   ：{VARIATIONS}')
print('=' * 72)

_finals = []
for _v in range(int(VARIATIONS)):
    _seed = (int(SEED) + _v) if int(SEED) > 0 else random.randint(1, 2**31 - 1)
    print(f'\n[{_v + 1}/{VARIATIONS}] seed = {_seed}')
    _video, _elapsed = _h3_generate(
        GEN_MODE, PROMPT, WIDTH, HEIGHT, LENGTH, _steps, _seed,
        tag=_h3_slug(_name, 40),
        first=_first, last=_last,
        ref_images=_refs_i, ref_videos=_refs_v, ref_audios=_refs_a,
        ref_image_size=REF_IMAGE_SIZE)
    _final = _h3_publish(_video, _name, _seed, {
        '鏡頭': _name,
        '時間': time.strftime('%Y-%m-%d %H:%M:%S'),
        '生成方式': GEN_MODE,
        'STEP 1 模式': _mode,
        '畫布': f'{WIDTH} x {HEIGHT}，{LENGTH} 幀（{LENGTH / 24:.2f} 秒）',
        '步數': _steps,
        'Seed': _seed,
        '首幀': str(_first) if _first else '',
        '尾幀': str(_last) if _last else '',
        '參考圖片': '、'.join(p.name for p in _refs_i),
        '參考影片': '、'.join(p.name for p in _refs_v),
        '參考音訊': '、'.join(p.name for p in _refs_a),
        '耗時': _h3_fmt(_elapsed),
        '提示詞': PROMPT,
    })
    _finals.append(_final)
    print(f'   完成，耗時 {_h3_fmt(_elapsed)}（每步 {_elapsed / _steps:.1f} 秒）')
    print(f'   成品：{_final}')
    if SHOW_PREVIEW:
        _h3_preview(_final)

print(f'\n全部完成，共 {len(_finals)} 支。成品與生成紀錄（同名 .txt）在：')
for _f in _finals:
    print(f'   {_f}')

if AUTO_DISCONNECT:
    _h3_disconnect()


### STEP 6G｜圖形介面（Gradio，**選用**）
想用圖形介面就執行這格（必須先執行 STEP 5），不想用就跳過，STEP 6 的表單照常可用。

- **免費版 Colab 請注意**：Colab 規定免費執行階段不能主要透過網頁介面操作，使用這格可能被中斷。付費方案且運算單元有餘額時不受此限。
- 勾選 `OPEN_IN_NEW_TAB`：這格會印出一個 `gradio.live` 網址和帳號密碼，點網址在新分頁登入使用。網址是公開的，請不要把帳號密碼分享出去。
- 取消勾選：介面直接出現在這格下方，不需要密碼。如果拿不到臨時網址，會自動改用內嵌模式。
- 介面上可以拖拉上傳首幀／尾幀，或從 `H3_Reference_Images` 挑現有圖片；上傳的圖會自動存一份到 `H3_Reference_Images`。
- 成品和 STEP 6 一樣存到 `AI_Video/Outputs`，「成品瀏覽」分頁可以回看影片和生成紀錄。
- 介面開著的期間執行階段會持續扣運算單元；用完請到「執行階段 → 中斷連線並刪除執行階段」。

In [ ]:
#@title STEP 6G｜圖形介面（Gradio）
#@markdown 必須先執行 STEP 5。執行這格後會出現圖形介面的網址。
#@markdown
#@markdown **另開分頁**：產生 `gradio.live` 臨時網址，用這格印出的帳號密碼登入。取消勾選則直接內嵌在這格下方（不需要密碼）。
OPEN_IN_NEW_TAB = True #@param {type:"boolean"}
#@markdown 密碼留空的話，每次執行會自動產生一組新的。
GUI_PASSWORD = '' #@param {type:"string"}

import builtins, random, re, secrets, shutil, subprocess, sys, time
from pathlib import Path

if '_h3_generate' not in globals():
    raise SystemExit('請先執行 STEP 5（每次重開執行階段都要跑 STEP 1、2、3、5），再執行 STEP 6G。')

try:
    import gradio as gr
except ImportError:
    print('安裝 Gradio 中（約半分鐘）…', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'gradio'], check=True)
    import gradio as gr

_GUI_CANVASES = [
    '1152 x 640 · 16:9 wide', '960 x 544 · 16:9 standard', '832 x 480 · 16:9 fast', '704 x 400 · 16:9 cheaper',
    '576 x 320 · 16:9 minimal', '640 x 1152 · 9:16 wide', '544 x 960 · 9:16 standard', '480 x 832 · 9:16 fast',
    '400 x 704 · 9:16 cheaper', '320 x 576 · 9:16 minimal', '768 x 768 · 1:1 full', '576 x 576 · 1:1 fast',
    '768 x 576 · 4:3', '576 x 768 · 3:4', '1344 x 544 · ~21:9', '1024 x 416 · ~21:9 cheaper']
_GUI_MODES = {'首尾幀生成（fl2va）': 'fl2va', '純文字生成（t2va）': 't2va', '參考素材生成（r2v）': 'r2v'}
_GUI_IMG = ('.png', '.jpg', '.jpeg', '.webp', '.bmp')
_GUI_VID = ('.mp4', '.mov', '.webm', '.mkv')
_GUI_AUD = ('.wav', '.mp3', '.flac', '.m4a', '.ogg')
_GUI_NONE = '（不使用）'


def _gui_ref_choices():
    names = []
    if H3_REF_DIR.is_dir():
        files = [p for p in H3_REF_DIR.iterdir() if p.is_file() and p.suffix.lower() in _GUI_IMG]
        names = [p.name for p in sorted(files, key=lambda p: p.stat().st_mtime, reverse=True)]
    return [_GUI_NONE] + names


def _gui_keep(path):
    """把在介面上傳的檔案存一份到 H3_Reference_Images，回傳那一份的路徑。"""
    src = Path(path)
    H3_REF_DIR.mkdir(parents=True, exist_ok=True)
    dest = H3_REF_DIR / src.name
    n = 2
    while dest.exists():
        if dest.stat().st_size == src.stat().st_size:
            return dest
        dest = H3_REF_DIR / f'{src.stem}_{n}{src.suffix}'
        n += 1
    shutil.copy2(src, dest)
    return dest


def _gui_frame(img, pick):
    """首幀／尾幀：從下拉選單挑的就直接用原檔，自己上傳的就存一份。"""
    if not img:
        return None
    if pick and pick != _GUI_NONE and Path(img).stem == Path(pick).stem and (H3_REF_DIR / pick).is_file():
        return H3_REF_DIR / pick
    return _gui_keep(img)


def _gui_pick_to_image(pick):
    if not pick or pick == _GUI_NONE:
        return None
    p = H3_REF_DIR / pick
    return str(p) if p.is_file() else None


def _gui_status(head, done):
    lines = [head]
    if done:
        lines += ['', '已完成的成品：'] + [f'  {p}' for p in done]
    return '\n'.join(lines)


def _gui_generate(shot, prompt, mode_label, first_img, first_pick, last_img, last_pick, ref_files, ref_size,
                  canvas, duration, steps, seed, variations, progress=gr.Progress()):
    gen = _GUI_MODES.get(mode_label, 'fl2va')
    prompt = (prompt or '').strip()
    if not prompt:
        raise gr.Error('請先填提示詞。')
    try:
        width, height = _h3_canvas(canvas)
    except ValueError as e:
        raise gr.Error(str(e)) from None
    first = last = None
    refs_i, refs_v, refs_a = [], [], []
    if gen == 'fl2va':
        if not first_img:
            raise gr.Error('首尾幀生成需要首幀：請上傳圖片，或從下拉選單挑一張。')
        first = _gui_frame(first_img, first_pick)
        last = _gui_frame(last_img, last_pick)
    elif gen == 'r2v':
        for f in ref_files or []:
            p = _gui_keep(getattr(f, 'name', f))
            ext = p.suffix.lower()
            if ext in _GUI_IMG:
                refs_i.append(p)
            elif ext in _GUI_VID:
                refs_v.append(p)
            elif ext in _GUI_AUD:
                refs_a.append(p)
            else:
                raise gr.Error(f'不支援的參考素材格式：{p.name}')
        if not (refs_i or refs_v or refs_a):
            raise gr.Error('參考素材生成至少要上傳一個參考檔案。')
        if len(refs_i) > 9 or len(refs_v) > 3 or len(refs_a) > 3:
            raise gr.Error('參考素材最多 9 張圖片、3 支影片、3 段音訊。')
    length = _h3_length(duration)
    steps = int(steps or 0) if int(steps or 0) > 0 else _h3_auto_steps(gen)
    name = (shot or '').strip() or _h3_slug(prompt, 40)
    mode = getattr(builtins, 'H3_MODE', 'standard')
    total = max(1, int(variations or 1))
    base_seed = int(seed or 0)
    done = []
    for v in range(total):
        this_seed = (base_seed + v) if base_seed > 0 else random.randint(1, 2**31 - 1)
        head = f'第 {v + 1}/{total} 支，seed = {this_seed}'
        yield _gui_status(head + '：生成中…（{0} x {1}，{2} 幀，{3} 步）'.format(width, height, length, steps), done), \
            (str(done[-1]) if done else None)
        seen = {'frac': None, 'step': ''}

        def _on_progress(elapsed, lines, _head=head, _seen=seen):
            for line in reversed(lines):
                m = re.search(r'(\d+)/(\d+) \[', line)
                if m and int(m.group(2)) > 0:
                    _seen['frac'] = min(int(m.group(1)) / int(m.group(2)), 1.0)
                    _seen['step'] = f'，取樣 {m.group(1)}/{m.group(2)}'
                    break
            progress(_seen['frac'] or 0.0,
                     desc=f'{_head}，已經過 {_h3_fmt(elapsed)}{_seen["step"] or "，載入模型中"}')

        try:
            video, elapsed = _h3_generate(
                gen, prompt, width, height, length, steps, this_seed,
                tag=_h3_slug(name, 40), first=first, last=last,
                ref_images=refs_i, ref_videos=refs_v, ref_audios=refs_a,
                ref_image_size=ref_size or 'match', on_progress=_on_progress)
        except Exception as e:
            raise gr.Error(f'生成失敗：{str(e)[:600]}') from None
        final = _h3_publish(video, name, this_seed, {
            '鏡頭': name,
            '時間': time.strftime('%Y-%m-%d %H:%M:%S'),
            '生成方式': gen,
            'STEP 1 模式': mode,
            '畫布': f'{width} x {height}，{length} 幀（{length / 24:.2f} 秒）',
            '步數': steps,
            'Seed': this_seed,
            '首幀': str(first) if first else '',
            '尾幀': str(last) if last else '',
            '參考圖片': '、'.join(p.name for p in refs_i),
            '參考影片': '、'.join(p.name for p in refs_v),
            '參考音訊': '、'.join(p.name for p in refs_a),
            '耗時': _h3_fmt(elapsed),
            '提示詞': prompt,
        })
        done.append(final)
        yield _gui_status(f'✅ {head}：完成，耗時 {_h3_fmt(elapsed)}（每步 {elapsed / steps:.1f} 秒）', done), str(final)
    yield _gui_status(f'✅ 全部完成，共 {len(done)} 支。', done), str(done[-1])


def _gui_history():
    files = []
    if H3_FINAL_DIR.is_dir():
        files = [p for p in H3_FINAL_DIR.rglob('*') if p.is_file() and p.suffix.lower() in _H3_VIDEO_EXT]
        files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return [p.relative_to(H3_FINAL_DIR).as_posix() for p in files[:200]]


def _gui_history_refresh():
    """重新讀成品清單，並直接顯示最新的一支。"""
    items = _gui_history()
    first = items[0] if items else None
    video, info = _gui_history_show(first)
    return gr.update(choices=items, value=first), video, info


def _gui_history_show(rel):
    if not rel:
        return None, ''
    p = (H3_FINAL_DIR / rel).resolve()
    if H3_FINAL_DIR.resolve() not in p.parents or not p.is_file():
        return None, '找不到這個檔案。'
    txt = p.with_suffix('.txt')
    return str(p), (txt.read_text(encoding='utf-8') if txt.exists() else '（這支沒有生成紀錄）')


_gui_mode_now = getattr(builtins, 'H3_MODE', 'standard')
with gr.Blocks(title='MiniMax H3 影片生成') as _h3_demo:
    gr.Markdown(f'## 🎬 MiniMax H3 影片生成\n'
                f'STEP 1 選的模式：`{_gui_mode_now}`（建議步數 {_h3_auto_steps("fl2va")}）。'
                f'成品會存到 `{H3_FINAL_DIR}`，旁邊有同名 `.txt` 生成紀錄。')
    with gr.Tab('生成'):
        with gr.Row():
            with gr.Column(scale=5):
                _g_shot = gr.Textbox(label='鏡頭代號（成品檔名用，例如 S01；可留空）')
                _g_prompt = gr.Textbox(label='提示詞（英文）', lines=6)
                _g_mode = gr.Radio(list(_GUI_MODES), value=list(_GUI_MODES)[0], label='生成方式')
                with gr.Column(visible=True) as _g_fl:
                    with gr.Row():
                        with gr.Column():
                            _g_first = gr.Image(label='首幀', type='filepath', sources=['upload', 'clipboard'], height=240)
                            _g_first_pick = gr.Dropdown(_gui_ref_choices(), value=_GUI_NONE,
                                                        label='或從 H3_Reference_Images 挑首幀')
                        with gr.Column():
                            _g_last = gr.Image(label='尾幀（選填）', type='filepath', sources=['upload', 'clipboard'], height=240)
                            _g_last_pick = gr.Dropdown(_gui_ref_choices(), value=_GUI_NONE,
                                                       label='或從 H3_Reference_Images 挑尾幀')
                    _g_refresh = gr.Button('重新整理圖片清單', size='sm')
                with gr.Column(visible=False) as _g_r2v:
                    _g_refs = gr.File(label='參考素材（圖片最多 9、影片 3、音訊 3；提示詞用 <Picture 1>、<Picture 2>… 依上傳順序對應）',
                                      file_count='multiple', type='filepath')
                    _g_ref_size = gr.Radio(['match', 'max'], value='match', label='參考圖尺寸（max 比較像，但比較慢）')
                with gr.Row():
                    _g_canvas = gr.Dropdown(_GUI_CANVASES, value='960 x 544 · 16:9 standard', label='畫布',
                                            allow_custom_value=True)
                    _g_duration = gr.Slider(2, 14, value=5, step=1, label='秒數')
                with gr.Row():
                    _g_steps = gr.Slider(0, 40, value=0, step=1, label='步數（0 = 依模式自動）')
                    _g_seed = gr.Number(value=0, precision=0, label='Seed（0 = 隨機）')
                    _g_vars = gr.Slider(1, 8, value=1, step=1, label='版本數')
                _g_go = gr.Button('生成影片', variant='primary')
            with gr.Column(scale=4):
                _g_status = gr.Textbox(label='狀態', lines=7, interactive=False)
                _g_video = gr.Video(label='最新成品', autoplay=True, loop=True, interactive=False)
    with gr.Tab('成品瀏覽'):
        with gr.Row():
            _g_hist = gr.Dropdown([], value=None, label='成品（新的在前）', scale=5)
            _g_hist_refresh = gr.Button('重新整理', scale=1)
        with gr.Row():
            _g_hist_video = gr.Video(label='播放', loop=True, interactive=False)
            _g_hist_info = gr.Textbox(label='生成紀錄', lines=16, interactive=False)

    _g_mode.change(lambda m: (gr.update(visible=_GUI_MODES.get(m) == 'fl2va'),
                              gr.update(visible=_GUI_MODES.get(m) == 'r2v')),
                   _g_mode, [_g_fl, _g_r2v])
    _g_first_pick.change(_gui_pick_to_image, _g_first_pick, _g_first)
    _g_last_pick.change(_gui_pick_to_image, _g_last_pick, _g_last)
    _g_refresh.click(lambda: (gr.update(choices=_gui_ref_choices()), gr.update(choices=_gui_ref_choices())),
                     None, [_g_first_pick, _g_last_pick])
    _g_go.click(_gui_generate,
                [_g_shot, _g_prompt, _g_mode, _g_first, _g_first_pick, _g_last, _g_last_pick, _g_refs, _g_ref_size,
                 _g_canvas, _g_duration, _g_steps, _g_seed, _g_vars],
                [_g_status, _g_video]).then(
                    _gui_history_refresh, None, [_g_hist, _g_hist_video, _g_hist_info])
    _g_hist_refresh.click(_gui_history_refresh, None, [_g_hist, _g_hist_video, _g_hist_info])
    _g_hist.input(_gui_history_show, _g_hist, [_g_hist_video, _g_hist_info])
    _h3_demo.load(_gui_history_refresh, None, [_g_hist, _g_hist_video, _g_hist_info])

# 重新執行這格時，先把上一次開的介面關掉。
if 'H3_GUI_DEMO' in globals():
    try:
        H3_GUI_DEMO.close()
    except Exception:
        pass
H3_GUI_DEMO = _h3_demo
_h3_demo.queue()
H3_FINAL_DIR.mkdir(parents=True, exist_ok=True)
H3_REF_DIR.mkdir(parents=True, exist_ok=True)
_gui_kw = dict(prevent_thread_lock=True, show_error=True,
               allowed_paths=[str(H3_REF_DIR), str(H3_FINAL_DIR), str(COMFY_DIR / 'output')])
_gui_kw.update(globals().get('_H3_GUI_LAUNCH_OVERRIDES', {}))

_gui_url = None
if OPEN_IN_NEW_TAB:
    # STEP 1 把 HF_HOME 設到雲端硬碟，Gradio 產生臨時網址用的 frpc 也會存在那裡；
    # 雲端硬碟重新掛載後那個檔案沒有執行權限，臨時網址會立刻失敗。改放在執行階段本機。
    try:
        import pathlib as _pl
        import gradio.blocks as _gr_blocks
        import gradio.tunneling as _gr_tunnel
        _frpc_dir = _pl.Path('/root/.cache/gradio_frpc')
        _frpc_dir.mkdir(parents=True, exist_ok=True)
        _gr_tunnel.BINARY_FOLDER = _frpc_dir
        _gr_tunnel.BINARY_PATH = str(_frpc_dir / _gr_tunnel.BINARY_FILENAME)
        _gr_blocks.BINARY_FOLDER, _gr_blocks.BINARY_PATH = _gr_tunnel.BINARY_FOLDER, _gr_tunnel.BINARY_PATH
    except Exception as _e:
        print(f'（無法調整 frpc 的位置：{_e}）')
    _gui_user = 'h3'
    _gui_pw = GUI_PASSWORD.strip() or secrets.token_urlsafe(9)
    for _gui_try in range(2):
        try:
            _, _, _gui_url = _h3_demo.launch(share=True, auth=(_gui_user, _gui_pw), inline=False, quiet=True, **_gui_kw)
        except Exception as _e:
            _gui_url = None
            print(f'（建立臨時網址時發生錯誤：{_e}）')
        if _gui_url:
            break
        try:
            _h3_demo.close()      # 沒拿到網址：先關掉，重試或改用內嵌模式
        except Exception:
            pass
        if _gui_try == 0:
            print(' 第一次沒有拿到臨時網址，再試一次……')
    print('=' * 72)
    if _gui_url:
        print(f' 圖形介面網址：{_gui_url}')
        print(f' 帳號：{_gui_user}')
        print(f' 密碼：{_gui_pw}')
        print(' 這個網址是公開的，只在這次執行階段有效；請不要分享帳號密碼。')
    else:
        print(' 沒有成功產生臨時網址，自動改用內嵌模式：面板會直接顯示在這格下方，不需要密碼。')
    print('=' * 72)
if not _gui_url:
    # 內嵌模式：告訴 Gradio 它在 Colab 代理後面的真正網址，圖片和影片預覽才載得出來。
    try:
        import socket
        from google.colab.output import eval_js
        _gui_port = 7860
        while _gui_port < 7900:
            with socket.socket() as _s:
                if _s.connect_ex(('127.0.0.1', _gui_port)) != 0:
                    break
            _gui_port += 1
        _gui_root = str(eval_js(f'google.colab.kernel.proxyPort({_gui_port})')).rstrip('/')
        if _gui_root.startswith('http'):
            _gui_kw.setdefault('server_port', _gui_port)
            _gui_kw.setdefault('root_path', _gui_root)
    except Exception as _e:
        print(f'（無法取得 Colab 代理網址，圖片預覽可能載不出來：{_e}）')
    _h3_demo.launch(share=False, inline=True, height=1200, **_gui_kw)
print('介面啟動後這格就算執行完了，介面會在背景持續運作；生成進度請看介面上的狀態欄。')


### STEP 7｜JSON 批次生成（進階，**選用**）
必須先執行 STEP 5。第一次執行若沒有 JSON，會產生範本並**停止，不會生成影片**。

- 開始前會先檢查**所有**場景；有任何一個填錯（例如圖片找不到）就全部不跑，並列出問題。
- 場景可以加 `"name": "S01"` 當成品檔名；圖片可以只填檔名。
- `SKIP_EXISTING`：已生成過、內容沒改的場景會跳過；場景內容或 STEP 1 模式有改就會重新生成。
- 單一場景失敗不會中斷整批，最後會列出失敗清單；修正後重跑即可。
- 成品在 `AI_Video/Outputs/<JSON 檔名>/`，每個場景的結果記在 JSON 旁邊的 `<檔名>_log.jsonl`。
- `AUTO_DISCONNECT`：整批跑完自動中斷執行階段。
- r2v 批次仍未實際用 GPU 測過，第一次請先少量測試。

In [ ]:
#@title STEP 7｜JSON 批次生成
"""
批次生成：讀一份 JSON 場景清單，一個場景生成一支影片。

JSON 是一個清單，每個場景一個物件，只有 prompt 必填：
[
  {
    "name": "S01",                     成品檔名用的鏡頭代號（選填）
    "prompt": "A red fox in a snowy pine forest at dawn",
    "gen_mode": "fl2va",               t2va / fl2va / r2v；沒填時，有 image 就是 fl2va，否則 t2va
    "image": "S01_start.png",          首幀；可以只填檔名（到 H3_Reference_Images 找）或完整路徑
    "last_image": "S01_end.png",       尾幀（選填）
    "ref_images": ["a.png", "b.png"],  r2v 用；ref_videos / ref_audios 同理
    "canvas": "832 x 480",             沒填用 DEFAULT_CANVAS
    "duration": 5,                     秒數，沒填用 DEFAULT_DURATION
    "steps": 20,                       沒填用 DEFAULT_STEPS
    "seed": 42                         0 或沒填 = 隨機
  }
]

開始生成前會先把所有場景檢查一遍；有任何場景填錯就全部不跑，直接列出問題。
每個場景的結果會寫進 JSON 旁邊的 <檔名>_log.jsonl。
"""
BATCH_JSON_PATH = '/content/drive/MyDrive/AI_Video/AEI_3D_Cache/H3_ComfyUI/batch_scenes.json' #@param {type:"string"}
DEFAULT_DURATION = 5 #@param {type:"slider", min:2, max:14, step:1}
#@markdown **DEFAULT_STEPS = 0**：依 STEP 1 選的模式自動決定步數。
DEFAULT_STEPS = 20 #@param {type:"slider", min:0, max:40, step:1}
DEFAULT_CANVAS = "832 x 480 · 16:9 fast (default)" #@param ['1152 x 640 · 16:9 wide', '960 x 544 · 16:9 standard', '832 x 480 · 16:9 fast (default)', '704 x 400 · 16:9 cheaper', '576 x 320 · 16:9 minimal', '640 x 1152 · 9:16 wide', '544 x 960 · 9:16 standard', '480 x 832 · 9:16 fast', '400 x 704 · 9:16 cheaper', '320 x 576 · 9:16 minimal', '768 x 768 · 1:1 full', '576 x 576 · 1:1 fast', '768 x 576 · 4:3', '576 x 768 · 3:4', '1344 x 544 · ~21:9', '1024 x 416 · ~21:9 cheaper'] {"allow-input": true}
#@markdown 已經生成過、內容沒改的場景就跳過（場景內容或模式有改就會重新生成）。
SKIP_EXISTING = True #@param {type:"boolean"}
#@markdown 勾選後，整批跑完會同步雲端硬碟並中斷執行階段（停止扣運算單元）。
AUTO_DISCONNECT = False #@param {type:"boolean"}

import builtins, hashlib, json, random, time
from pathlib import Path

if '_h3_generate' not in globals():
    raise SystemExit('請先執行 STEP 5（每次重開執行階段都要跑 STEP 1、2、3、5），再執行 STEP 7。')

_json_path = Path(BATCH_JSON_PATH)
if not _json_path.exists():
    _json_path.parent.mkdir(parents=True, exist_ok=True)
    _template = [
        {'name': 'fox', 'prompt': 'A red fox in a snowy pine forest at dawn, slow dolly push-in, snow crunching underfoot.',
         'canvas': DEFAULT_CANVAS, 'duration': DEFAULT_DURATION, 'seed': 42},
        {'name': 'market', 'prompt': 'A busy night market, neon signs reflecting in puddles, sizzling street food, ambient chatter.',
         'canvas': DEFAULT_CANVAS, 'duration': DEFAULT_DURATION, 'seed': 7},
    ]
    _json_path.write_text(json.dumps(_template, indent=2, ensure_ascii=False), encoding='utf-8')
    raise SystemExit(f'已建立 JSON 範例：{_json_path}\n請先修改內容，再執行 STEP 7（尚未生成影片）。')

try:
    _scenes = json.loads(_json_path.read_text(encoding='utf-8'))
except json.JSONDecodeError as _e:
    raise SystemExit(f'JSON 格式有誤（第 {_e.lineno} 行、第 {_e.colno} 個字附近）：{_e.msg}') from None
if not isinstance(_scenes, list):
    raise SystemExit('JSON 最外層必須是清單 [ ... ]。')

_mode = getattr(builtins, 'H3_MODE', 'standard')
_out_dir = COMFY_DIR / 'output'
_final_dir = H3_FINAL_DIR / _h3_slug(_json_path.stem, 40)
_log_path = _json_path.with_name(_json_path.stem + '_log.jsonl')

# ── 先把所有場景檢查一遍 ──
_plans, _errors = [], []
for _i, _sc in enumerate(_scenes):
    _n = _i + 1
    try:
        if not isinstance(_sc, dict):
            raise ValueError('這一項不是 { ... } 物件。')
        _prompt = str(_sc.get('prompt') or '').strip()
        if not _prompt:
            raise ValueError('prompt 是空的。')
        _w, _h = _h3_canvas(_sc.get('canvas') or DEFAULT_CANVAS)
        _length = _h3_length(int(_sc.get('duration') or DEFAULT_DURATION))
        _first = _h3_resolve(_sc.get('image'), '首幀圖片')
        _last = _h3_resolve(_sc.get('last_image'), '尾幀圖片')
        _gen = _sc.get('gen_mode') or ('fl2va' if _first else 't2va')
        if _gen not in ('t2va', 'fl2va', 'r2v'):
            raise ValueError(f'不支援的 gen_mode：{_gen}')
        _ri = _h3_resolve_list(_sc.get('ref_images'), '參考圖片') if _gen == 'r2v' else []
        _rv = _h3_resolve_list(_sc.get('ref_videos'), '參考影片') if _gen == 'r2v' else []
        _ra = _h3_resolve_list(_sc.get('ref_audios'), '參考音訊') if _gen == 'r2v' else []
        if _gen == 'fl2va' and not _first:
            raise ValueError('fl2va 必須指定 image（首幀）。')
        if _gen != 'fl2va':
            _first = _last = None
        if _gen == 'r2v' and not (_ri or _rv or _ra):
            raise ValueError('r2v 至少要一個參考素材（ref_images / ref_videos / ref_audios）。')
        if len(_ri) > 9 or len(_rv) > 3 or len(_ra) > 3:
            raise ValueError('r2v 最多 9 張圖片、3 支影片、3 段音訊。')
        _steps = int(_sc.get('steps') or 0) or (int(DEFAULT_STEPS) if int(DEFAULT_STEPS) > 0 else _h3_auto_steps(_gen))
        _raw_seed = int(_sc.get('seed') or 0)
        _size = str(_sc.get('ref_image_size') or 'match')
        _name = str(_sc.get('name') or '').strip() or _h3_slug(_prompt, 30)
        # 場景內容的指紋：內容或模式一改，指紋就不同，就會重新生成。
        _key = hashlib.sha1(json.dumps(
            [_mode, _gen, _prompt, _w, _h, _length, _steps, _raw_seed, str(_first), str(_last),
             [str(p) for p in _ri], [str(p) for p in _rv], [str(p) for p in _ra], _size],
            ensure_ascii=False).encode('utf-8')).hexdigest()[:8]
        _plans.append(dict(n=_n, name=_name, prompt=_prompt, gen=_gen, w=_w, h=_h, length=_length,
                           steps=_steps, seed=_raw_seed, first=_first, last=_last,
                           ri=_ri, rv=_rv, ra=_ra, size=_size, key=_key))
    except (ValueError, TypeError) as _e:
        _errors.append(f'  場景 {_n}：{_e}')

if _errors:
    raise SystemExit('批次檔有問題，還沒開始生成，請先修正：\n' + '\n'.join(_errors))

print('=' * 72)
print(f' 批次檔 ：{_json_path}')
print(f' 場景數 ：{len(_plans)}（STEP 1 模式：{_mode}）')
print(f' 成品   ：{_final_dir}')
print('=' * 72)


def _h3_batch_log(**row):
    row['time'] = time.strftime('%Y-%m-%d %H:%M:%S')
    try:
        with _log_path.open('a', encoding='utf-8') as _f:
            _f.write(json.dumps(row, ensure_ascii=False) + '\n')
    except OSError:
        pass


def _h3_batch_existing(key):
    for _d in (_out_dir / 'video', _out_dir):
        if _d.is_dir():
            _hit = next(iter(_d.glob(f'H3_batch_*_{key}_*')), None)
            if _hit:
                return _hit
    return None


_done, _skipped, _failed = [], [], []
_t_all = time.time()
for _p in _plans:
    _label = f'[{_p["n"]}/{len(_plans)}] {_p["name"]}'
    if SKIP_EXISTING:
        _hit = _h3_batch_existing(_p['key'])
        if _hit:
            print(f'\n{_label}：之前已完成，跳過（{_hit.name}）')
            _skipped.append(_p['n'])
            _h3_batch_log(scene=_p['n'], name=_p['name'], status='skipped', file=str(_hit))
            continue
    _seed = _p['seed'] if _p['seed'] > 0 else random.randint(1, 2**31 - 1)
    print(f'\n{_label}：{_p["gen"]}，{_p["w"]} x {_p["h"]}，{_p["length"]} 幀，{_p["steps"]} 步，seed = {_seed}')
    print(f'   {_p["prompt"][:80]}')
    try:
        _video, _elapsed = _h3_generate(
            _p['gen'], _p['prompt'], _p['w'], _p['h'], _p['length'], _p['steps'], _seed,
            tag=f'batch_{_p["n"]:03d}_{_p["key"]}_{_h3_slug(_p["name"], 24)}',
            first=_p['first'], last=_p['last'],
            ref_images=_p['ri'], ref_videos=_p['rv'], ref_audios=_p['ra'],
            ref_image_size=_p['size'], poll=5)
        _final = _h3_publish(_video, f'{_p["n"]:03d}_{_p["name"]}', _seed, {
            '場景': _p['n'],
            '鏡頭': _p['name'],
            '時間': time.strftime('%Y-%m-%d %H:%M:%S'),
            '生成方式': _p['gen'],
            'STEP 1 模式': _mode,
            '畫布': f'{_p["w"]} x {_p["h"]}，{_p["length"]} 幀（{_p["length"] / 24:.2f} 秒）',
            '步數': _p['steps'],
            'Seed': _seed,
            '首幀': str(_p['first']) if _p['first'] else '',
            '尾幀': str(_p['last']) if _p['last'] else '',
            '參考圖片': '、'.join(x.name for x in _p['ri']),
            '參考影片': '、'.join(x.name for x in _p['rv']),
            '參考音訊': '、'.join(x.name for x in _p['ra']),
            '耗時': _h3_fmt(_elapsed),
            '提示詞': _p['prompt'],
        }, dest_dir=_final_dir)
        _done.append(_final)
        print(f'   完成，耗時 {_h3_fmt(_elapsed)}：{_final.name}')
        _h3_batch_log(scene=_p['n'], name=_p['name'], status='done', seed=_seed,
                      seconds=round(_elapsed), file=str(_final))
    except Exception as _e:
        _failed.append((_p['n'], _p['name'], str(_e)[:300]))
        print(f'   ✗ 失敗，繼續下一個場景：{str(_e)[:300]}')
        _h3_batch_log(scene=_p['n'], name=_p['name'], status='failed', seed=_seed, error=str(_e)[:500])

print('\n' + '=' * 72)
print(f' 批次結束，總耗時 {_h3_fmt(time.time() - _t_all)}：'
      f'完成 {len(_done)}、跳過 {len(_skipped)}、失敗 {len(_failed)}')
for _f in _done:
    print(f'   ✓ {_f}')
for _n, _name, _msg in _failed:
    print(f'   ✗ 場景 {_n}（{_name}）：{_msg}')
if _failed:
    print(' 失敗的場景修正後重跑 STEP 7 即可，已完成的會自動跳過。')
print(f' 紀錄檔：{_log_path}')
print('=' * 72)

if AUTO_DISCONNECT:
    _h3_disconnect()


### STEP 8｜查看 ComfyUI 錯誤日誌
如果影片生成失敗，按 ▶，把最後的錯誤訊息複製出來協助排查。

In [ ]:
#@title STEP 8｜錯誤日誌

import builtins
from pathlib import Path

LOG = Path(getattr(builtins, 'AEI_COMFY_LOG', '/content/drive/MyDrive/AI_Video/AEI_ComfyUI/comfyui.log'))
TAIL_LINES = 80  #@param {type:"slider", min:20, max:500, step:20}

if not LOG.exists():
    print(f'  Log not found: {LOG}')
else:
    lines = LOG.read_text(errors='replace').splitlines()
    n = min(TAIL_LINES, len(lines))
    print(f'=== last {n} lines of {LOG} ===')
    for ln in lines[-n:]:
        print(ln)
    print('=== end ===')


### STEP 8.5｜短片工作流程驗證（選用）
這格**真的會執行推論並消耗 GPU 額度**，只在除錯需要時使用。

In [ ]:
#@title STEP 8.5｜工作流程短片驗證

import os, sys, time, json, uuid, urllib.request, urllib.error, builtins
from pathlib import Path
import requests

# --- self-contained workflow driver (works independently of STEP 5) ---
URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
OUT_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR', Path('/content/drive/MyDrive/AI_Video/AEI_ComfyUI'))) / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _poll_until_ready(url, timeout=10):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                r.read()
            return True
        except Exception:
            time.sleep(1)
    return False

if not _poll_until_ready(URL + '/system_stats'):
    raise SystemExit('ComfyUI is not running. Re-run STEP 3.')

# STEP 6/7 look up UNET/CLIP/VVAE/AVAE from H3_MODE; replicate that here.
_MODE_SMOKE = getattr(builtins, 'H3_MODE', 'standard')
_SEG = _MODE_SMOKE.split('+') if '+' in _MODE_SMOKE else [_MODE_SMOKE]
_TURBO = _SEG[-1]
_SPECTRUM = 'spectrum' in _SEG
_FP8 = _MODE_SMOKE.startswith('fp8_')
_LIGHTX2V = _TURBO.startswith('lightx2v_')
_ERSDE = _TURBO.endswith('_ersde')

UNET = 'minimax_h3_fl2va_pruned_fp8_scaled.safetensors' if _FP8 else 'minimax_h3_fl2va_pruned_int8_convrot.safetensors'
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'
SAMPLER = 'er_sde' if _ERSDE else 'res_multistep'

# Default steps for the active mode. Mirrors STEP 1's _MODE_DEFAULT_STEPS.
_STEPS_MAP = {
    'standard': 20, 'turbo_lora': 10, 'turbo_lora_sage': 8,
    'spectrum': 16, 'spectrum+turbo_lora': 12, 'spectrum+turbo_lora_sage': 10,
    'fp8_dense': 16, 'fp8_dense+turbo_lora': 10, 'fp8_dense+turbo_lora_sage': 8,
    'fp8_dense+spectrum': 16, 'fp8_dense+spectrum+turbo_lora': 12, 'fp8_dense+spectrum+turbo_lora_sage': 10,
    'lightx2v_4step': 4, 'lightx2v_8step': 8,
    'lightx2v_4step_ersde': 5, 'lightx2v_8step_ersde': 9,
    'spectrum+lightx2v_4step': 6, 'spectrum+lightx2v_8step': 10,
    'spectrum+lightx2v_4step_ersde': 7, 'spectrum+lightx2v_8step_ersde': 11,
    'fp8_dense+lightx2v_4step': 4, 'fp8_dense+lightx2v_8step': 8,
    'fp8_dense+lightx2v_4step_ersde': 5, 'fp8_dense+lightx2v_8step_ersde': 9,
    'fp8_dense+spectrum+lightx2v_4step': 6, 'fp8_dense+spectrum+lightx2v_8step': 10,
    'fp8_dense+spectrum+lightx2v_4step_ersde': 7, 'fp8_dense+spectrum+lightx2v_8step_ersde': 11,
}
DEFAULT_STEPS = _STEPS_MAP.get(_MODE_SMOKE, 8)
# Same Larryvrh override as STEP 1/6/7.
if _MODE_SMOKE in ('turbo_lora', 'turbo_lora_sage',
                   'spectrum+turbo_lora', 'spectrum+turbo_lora_sage',
                   'fp8_dense+turbo_lora', 'fp8_dense+turbo_lora_sage',
                   'fp8_dense+spectrum+turbo_lora',
                   'fp8_dense+spectrum+turbo_lora_sage') and         getattr(__import__('builtins'), 'H3_TURBO_BACKEND', 'drbaph') == 'larryvrh':
    DEFAULT_STEPS = {'turbo_lora': 6, 'turbo_lora_sage': 6,
                     'spectrum+turbo_lora': 8, 'spectrum+turbo_lora_sage': 8,
                     'fp8_dense+turbo_lora': 6, 'fp8_dense+turbo_lora_sage': 6,
                     'fp8_dense+spectrum+turbo_lora': 8,
                     'fp8_dense+spectrum+turbo_lora_sage': 8,
                    }[_MODE_SMOKE]

# --- user-tunable params (small defaults for fast smoke testing) ---
# Default mode is lightx2v_4step because it finishes in ~60-90s on L4+Pro+
# (4 sampling steps + a small 56-frame clip at 480x832). NOTE: the eval
# notes 4-step lightx2v has the most trajectory-deviation of any tested
# config; for visual quality, switch to 'lightx2v_8step' (~2-3 min)
# or 'spectrum+turbo_lora_sage' (~3-4 min, the eval best config).
MODE_TEST = "use_h3_mode"  #@param ['standard', 'turbo_lora', 'turbo_lora_sage', 'spectrum', 'spectrum+turbo_lora', 'spectrum+turbo_lora_sage', 'fp8_dense', 'fp8_dense+turbo_lora', 'fp8_dense+turbo_lora_sage', 'fp8_dense+spectrum', 'fp8_dense+spectrum+turbo_lora', 'fp8_dense+spectrum+turbo_lora_sage', 'lightx2v_4step', 'lightx2v_8step', 'lightx2v_4step_ersde', 'lightx2v_8step_ersde', 'spectrum+lightx2v_4step', 'spectrum+lightx2v_8step', 'spectrum+lightx2v_4step_ersde', 'spectrum+lightx2v_8step_ersde', 'fp8_dense+lightx2v_4step', 'fp8_dense+lightx2v_8step', 'fp8_dense+lightx2v_4step_ersde', 'fp8_dense+lightx2v_8step_ersde', 'fp8_dense+spectrum+lightx2v_4step', 'fp8_dense+spectrum+lightx2v_8step', 'fp8_dense+spectrum+lightx2v_4step_ersde', 'fp8_dense+spectrum+lightx2v_8step_ersde', 'use_h3_mode']
# When MODE_TEST = 'use_h3_mode', this cell uses H3_MODE from STEP 1 unchanged.
# Otherwise MODE_TEST overrides the workflow but NOT the diffusion file
# selection in STEP 2 (the wrong UNET file will fail with a clear
# 'unet not found' error if MODE_TEST references a model that wasn't
# downloaded in STEP 2).

PROMPT = "A woman with short gray hair speaks to the camera in a small studio."  #@param {type:"string"}
# 56 frames = 2.33 s at 24 fps; 480x832 is the "fast" canvas (lowest VRAM).
LENGTH = 56  #@param {type:"slider", min:39, max:124, step:1}
# Colab form param needs a literal number — see STEP 6 for the full rationale.
_STEPS_SLIDER_DEFAULT = DEFAULT_STEPS
STEPS = 4  #@param {type:"slider", min:4, max:40, step:1}
SEED = 424242  #@param {type:"integer"}
WIDTH = 480  #@param {type:"slider", min:384, max:1344, step:32}
HEIGHT = 832  #@param {type:"slider", min:384, max:1344, step:32}
# Poll cadence: 3s is the previous version default; raise to 6s for slow modes
# (spectrum+lightx2v_*_ersde can take 5-10 min on a 56-frame clip).
POLL_SECONDS = 3  #@param {type:"slider", min:1, max:15, step:1}
TIMEOUT_MIN = 15  #@param {type:"slider", min:5, max:60, step:1}

# Override H3_MODE for this cell if MODE_TEST != 'use_h3_mode'.
_EFFECTIVE_MODE = _MODE_SMOKE if MODE_TEST == 'use_h3_mode' else MODE_TEST
_EFF_SEG = _EFFECTIVE_MODE.split('+') if '+' in _EFFECTIVE_MODE else [_EFFECTIVE_MODE]
_EFF_TURBO = _EFF_SEG[-1]
_EFF_SPECTRUM = 'spectrum' in _EFF_SEG
_EFF_FP8 = _EFFECTIVE_MODE.startswith('fp8_')
_EFF_LIGHTX2V = _EFF_TURBO.startswith('lightx2v_')
_EFF_ERSDE = _EFF_TURBO.endswith('_ersde')

# Per-mode defaults if user set MODE_TEST to a different mode.
if MODE_TEST != 'use_h3_mode':
    UNET = 'minimax_h3_fl2va_pruned_fp8_scaled.safetensors' if _EFF_FP8 else 'minimax_h3_fl2va_pruned_int8_convrot.safetensors'
    SAMPLER = 'er_sde' if _EFF_ERSDE else 'res_multistep'
    STEPS = _STEPS_MAP.get(_EFFECTIVE_MODE, DEFAULT_STEPS)
    DEFAULT_STEPS = STEPS

# --- inline _build_workflow (standalone subset of the workflow builder) ---
import re as _re
def _slug(s, maxlen=40):
    s = _re.sub(r'[^a-zA-Z0-9_\\-]+', '-', s).strip('-')
    s = _re.sub(r'-+', '-', s)
    return s[:maxlen].strip('-') or 'untitled'

TURBO_LORA = 'minimax_h3_turbo_4step_ema_ckpt500_pruned_comfyui.safetensors'
LIGHTX2V_LORA = 'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'
_EFF_SAGE = _EFF_TURBO == 'turbo_lora_sage'

# Pre-flight check: warn if the user changed MODE_TEST to a mode whose
# LoRA / diffusion / custom nodes weren't downloaded in STEP 2. We
# don't abort — ComfyUI's own error messages are clear — but the
# warning makes the cause obvious in the smoke-test output.
LORA_DIR_CHECK = OUT_DIR.parent / 'models' / 'loras'
WARNINGS = []
if _EFF_LIGHTX2V or _EFF_TURBO in ('turbo_lora', 'turbo_lora_sage'):
    _needed_lora = LIGHTX2V_LORA if _EFF_LIGHTX2V else 'minimax_h3_turbo_4step_ema_ckpt500_pruned_comfyui.safetensors'
    if not (LORA_DIR_CHECK / _needed_lora).exists():
        WARNINGS.append(f'  ! LoRA not found: {LORA_DIR_CHECK / _needed_lora}')
        WARNINGS.append(f'    → Re-run STEP 2 to download it (the chosen mode needs the LoRA).')
if _EFF_SAGE and not (OUT_DIR.parent / 'custom_nodes' / 'ComfyUI-KJNodes').exists():
    WARNINGS.append(f'  ! kjnodes not installed (ComfyUI-KJNodes)')
    WARNINGS.append(f'    → Re-run STEP 3 with MODE = turbo_lora_sage to install it.')
if _EFF_SPECTRUM and not (OUT_DIR.parent / 'custom_nodes' / 'ComfyUI-Spectrum-MiniMax-H3').exists():
    WARNINGS.append(f'  ! Spectrum not installed (ComfyUI-Spectrum-MiniMax-H3)')
    WARNINGS.append(f'    → Re-run STEP 3 with a spectrum+ mode to install it.')
# Sage is a soft dependency. The kjnodes SageAttention node silently
# no-ops if `sageattn` isn't pip-installed, so we don't warn — but the
# user should know that without `pip install sageattn` the sage variant
# runs at parity with the non-sage variant. Print a hint only if the
# user picked a sage mode.
try:
    import sageattention  # noqa: F401
    from sageattention.core import get_cuda_arch_versions  # noqa: F401
    _HAS_SAGE = True
except (ImportError, AttributeError):
    _HAS_SAGE = False
if _EFF_SAGE and not _HAS_SAGE:
    print('  ℹ thu-ml sageattention package not installed (or too old — '
          'PyPI 1.0.6 is missing get_cuda_arch_versions).')
    print('    (Fix: `pip install git+https://github.com/thu-ml/SageAttention@main`, '
          'then re-run STEP 3.)')
_UNET_PATH = OUT_DIR.parent / 'models' / 'diffusion_models' / UNET
if not _UNET_PATH.exists():
    WARNINGS.append(f'  ! Diffusion model not found: {_UNET_PATH}')
    WARNINGS.append(f'    → Re-run STEP 2 to download the fp8_scaled or int8_convrot diffusion model.')
for w in WARNINGS:
    print(w)
if WARNINGS:
    print('  (continuing — ComfyUI will report the missing-file error itself if any of the above are fatal)')



def _build(mode, prompt, w, h, length, steps, seed, lora_name, lora_strength,
           sigma_v, sigma_a, sampler, has_lora, has_sage, has_spectrum, has_turbo,
           use_larryvrh_turbo=False, larryvrh_low_vram=False,
           use_sol=False, scheduler_name='simple'):
    p = {}
    p["6"]  = {"class_type": "UNETLoader",   "inputs": {"unet_name": UNET, "weight_dtype": "default"}}
    p["13"] = {"class_type": "CLIPLoader",   "inputs": {"clip_name": CLIP, "type": "minimax", "device": "default"}}
    p["11"] = {"class_type": "VAELoader",    "inputs": {"vae_name": VVAE}}
    p["24"] = {"class_type": "VAELoader",    "inputs": {"vae_name": AVAE}}
    inputs = {"clip": ["13", 0], "vae": ["11", 0],
              "width": w, "height": h, "length": length, "prompt": prompt}
    p["104"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": inputs}
    _model_out = ["6", 0]
    p["17"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": sampler}}
    if use_larryvrh_turbo and has_turbo:
        # Larryvrh Turbo LoRA node replaces the LoraLoader + SigmaShift
        # pair. The Turbo Sampler handles SigmaShift internally.
        p["40"] = {"class_type": "MiniMaxH3TurboLoRA",
                   "inputs": {"model": _model_out,
                              "lora_name": lora_name,
                              "strength": lora_strength,
                              "low_vram": larryvrh_low_vram}}
        _model_out = ["40", 0]
        if has_sage:
            p["42"] = {"class_type": "MiniMaxH3MemoryEfficientSageAttentionPatch",
                       "inputs": {"model": _model_out, "sage_attention": 2}}
            _model_out = ["42", 0]
        if use_sol:
            p["43"] = {"class_type": "MiniMaxH3MemoryEfficientSolAttentionPatch",
                       "inputs": {"model": _model_out, "enabled": True,
                                  "tau": 1.3, "min_tokens": 4096, "strict": False,
                                  "thresh_type": "diag", "int8_qk": False, "int8_pv": False,
                                  "sink_conditioning": "exact_kv", "dense_blocks": ""}}
            _model_out = ["43", 0]
        p["17"] = {"class_type": "MiniMaxH3TurboSampler", "inputs": {}}
    elif has_lora or has_turbo:
        p["40"] = {"class_type": "LoraLoaderModelOnly",
                   "inputs": {"model": _model_out, "lora_name": lora_name, "strength_model": lora_strength}}
        _model_out = ["40", 0]
        p["41"] = {"class_type": "MiniMaxH3SigmaShift",
                   "inputs": {"model": _model_out, "shift_video": sigma_v, "shift_audio": sigma_a}}
        _model_out = ["41", 0]
        if has_sage:
            p["42"] = {"class_type": "MiniMaxH3MemoryEfficientSageAttentionPatch",
                       "inputs": {"model": _model_out, "sage_attention": 2}}
            _model_out = ["42", 0]
        if use_sol:
            p["43"] = {"class_type": "MiniMaxH3MemoryEfficientSolAttentionPatch",
                       "inputs": {"model": _model_out, "enabled": True,
                                  "tau": 1.3, "min_tokens": 4096, "strict": False,
                                  "thresh_type": "diag", "int8_qk": False, "int8_pv": False,
                                  "sink_conditioning": "exact_kv", "dense_blocks": ""}}
            _model_out = ["43", 0]
        p["17"] = {"class_type": "KSamplerSelect","inputs": {"sampler_name": sampler}}
    if has_spectrum:
        _spec_on = sampler in ('euler', 'res_multistep', 'res_multistep_cfg_pp')
        p["50"] = {"class_type": "SpectrumApplyMiniMaxH3",
                   "inputs": {"model": _model_out,
                              "enabled": _spec_on,
                              "blend_weight": 0.50, "degree": 1, "ridge_lambda": 0.10,
                              "window_size": 2.0, "flex_window": 0.75,
                              "warmup_steps": 1, "tail_actual_steps": 1, "max_history": 8,
                              "debug": False, "history_storage": "system_ram",
                              "bootstrap_first_forecast": _spec_on}}
        _model_out = ["50", 0]
    p["16"] = {"class_type": "BasicGuider",   "inputs": {"model": _model_out, "conditioning": ["104", 0]}}
    # p["17"] is set inside the lora/turbo branch above (KSamplerSelect or
    # MiniMaxH3TurboSampler).
    p["9"]  = {"class_type": "BasicScheduler","inputs": {"model": _model_out, "scheduler": scheduler_name,
                                                          "steps": steps, "denoise": 1}}
    p["15"] = {"class_type": "RandomNoise",   "inputs": {"noise_seed": seed}}
    p["14"] = {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["15", 0], "guider": ["16", 0], "sampler": ["17", 0],
                          "sigmas": ["9", 0], "latent_image": ["104", 1]}}
    p["10"] = {"class_type": "VAEDecode",      "inputs": {"samples": ["14", 0], "vae": ["11", 0]}}
    p["23"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["14", 0], "vae": ["24", 0]}}
    p["91"] = {"class_type": "CreateVideo",    "inputs": {"images": ["10", 0], "audio": ["23", 0], "fps": 24}}
    # Filename prefix uses video/<mode>... so ComfyUI saves into output/video/ and
    # /history reports just the basename. We probe both locations below.
    slug = _slug(prompt, maxlen=40)
    ts = int(time.time())
    prefix = (f'video/H3_smoke_{mode}_{slug}_{w}x{h}_f{length}'
              f'_s{steps}_seed{seed}_{ts}')
    p["92"] = {"class_type": "SaveVideo",      "inputs": {"video": ["91", 0],
                                                            "filename_prefix": prefix,
                                                            "format": "auto", "codec": "auto"}}
    return {"prompt": p}

# Derive the right LoRA / strength / SigmaShift for the effective mode.
if _EFF_LIGHTX2V:
    _lora_name = LIGHTX2V_LORA
    _lora_strength = 0.75 if _EFF_ERSDE else 1.0
    _sigma_v, _sigma_a = (12.0, 3.0) if _EFF_ERSDE else (12.0, 6.0)
elif _EFF_TURBO in ('turbo_lora', 'turbo_lora_sage'):
    _lora_name = TURBO_LORA
    _lora_strength = 1.0
    _sigma_v, _sigma_a = 12.0, 6.0
else:
    _lora_name = TURBO_LORA  # unused
    _lora_strength = 1.0
    _sigma_v, _sigma_a = 12.0, 6.0

_has_turbo = _EFF_TURBO in ('turbo_lora', 'turbo_lora_sage')
_has_lora = _EFF_LIGHTX2V or _has_turbo
_has_sage = _EFF_TURBO == 'turbo_lora_sage'
# Larryvrh backend: same conditions as the main builder (use it on T4 / sub-16 GB).
_larryvrh_smoke = (_has_turbo and not _EFF_LIGHTX2V
                  and getattr(__import__('builtins'),
                              'H3_TURBO_BACKEND', 'drbaph') == 'larryvrh')
_larryvrh_low_vram_smoke = getattr(__import__('builtins'), 'H3_LOW_VRAM', False)
# Attention: 'sage' / 'sol' / 'sage+sol' / 'none'. We always run sage
# when sage is in the chain (covers the turbo_lora_sage path
# automatically), and we run sol when the active attention mode
# includes it.
_attn = getattr(__import__('builtins'), 'H3_ATTENTION', 'sage')
_use_sol_smoke = _attn in ('sol', 'sage+sol')

print('='*72)
print(f'  STEP 8.5 smoke test  (mode={_EFFECTIVE_MODE})')
print('='*72)
print(f'  diffusion   : {UNET}')
print(f'  lora        : {_lora_name if _has_lora else "(none)"}  strength={_lora_strength if _has_lora else "-"}')
print(f'  sigma_shift : video={_sigma_v}, audio={_sigma_a}')
print(f'  sampler     : {SAMPLER}')
print(f'  sage        : {_has_sage}')
print(f'  spectrum    : {_EFF_SPECTRUM}  ({"active" if _EFF_SPECTRUM and SAMPLER in ("euler", "res_multistep", "res_multistep_cfg_pp") else "pass-through" if _EFF_SPECTRUM else "n/a"})')
print(f'  prompt      : {PROMPT!r}')
print(f'  canvas      : {WIDTH}x{HEIGHT}  length={LENGTH} frames  steps={STEPS}  seed={SEED}')

wf = _build('t2va', PROMPT, WIDTH, HEIGHT, LENGTH, STEPS, SEED,
            _lora_name, _lora_strength, _sigma_v, _sigma_a, SAMPLER,
            _has_lora, _has_sage, _EFF_SPECTRUM, _has_turbo,
            use_larryvrh_turbo=_larryvrh_smoke,
            larryvrh_low_vram=_larryvrh_low_vram_smoke,
            use_sol=_use_sol_smoke)

client_id = str(uuid.uuid4())
t0 = time.time()
print(f'  Queueing workflow (client_id={client_id[:8]})...')
r = requests.post(URL + '/prompt', json={"prompt": wf["prompt"], "client_id": client_id}, timeout=60)
if r.status_code != 200:
    raise SystemExit(f'ComfyUI rejected workflow: HTTP {r.status_code}\n{r.text[:1500]}')
prompt_id = r.json()['prompt_id']
print(f'  Queued as {prompt_id[:8]}. Polling /history every {POLL_SECONDS}s (timeout {TIMEOUT_MIN} min)...')

deadline = time.time() + TIMEOUT_MIN * 60
while time.time() < deadline:
    hr = requests.get(f'{URL}/history/{prompt_id}', timeout=10)
    if hr.status_code == 200 and prompt_id in hr.json():
        entry = hr.json()[prompt_id]
        if entry.get('status', {}).get('completed'):
            break
        if entry.get('status', {}).get('status_str') == 'error':
            raise SystemExit('ComfyUI 發生執行錯誤：' + json.dumps(entry.get('status', {}).get('messages', []), ensure_ascii=False)[:2000])
    time.sleep(POLL_SECONDS)
else:
    raise SystemExit(f'Workflow did not finish within {TIMEOUT_MIN} min. Check STEP 8 for log tail.')

def _fmt(seconds):
    s = int(seconds)
    m, s = divmod(s, 60)
    h, m = divmod(m, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

elapsed = time.time() - t0
print(f'  Completed in {_fmt(elapsed)}')

# Find the output video. ComfyUI strips the dir prefix from /history, so
# 'video/H3_...' becomes just 'H3_...'; we probe both locations.
out_files = []
for k, v in entry.get('outputs', {}).items():
    for f in v.get('videos', []) + v.get('files', []):
        out_files.append(f.get('filename', ''))
out_files = [f for f in out_files if f]
print(f'  /history reported {len(out_files)} file(s): {out_files}')

# Probe locations
found = None
for cand in out_files:
    for path in (OUT_DIR / cand, OUT_DIR / 'video' / cand):
        if path.exists() and path.stat().st_size > 0:
            found = path
            break
    if found:
        break

if not found:
    # last-resort recursive glob
    matches = list(OUT_DIR.glob('**/H3_smoke_*.mp4'))
    matches = [m for m in matches if (time.time() - m.stat().st_mtime) < 600]
    if matches:
        found = max(matches, key=lambda p: p.stat().st_mtime)

if found:
    sz = found.stat().st_size / 1024**2
    print(f'  ✓ Video found: {found}  ({sz:.1f} MB)')
    print(f'  To preview: right-click in the file browser, or copy the path.')
    print(f'  Full path: {found}')
else:
    print(f'  ✗ Video not found. /history said: {out_files}')
    print(f'    Looked in: {OUT_DIR}  and  {OUT_DIR}/video/')
    print(f'    Run STEP 8 to check the ComfyUI log.')
